# Zoning PDF Pilot — Greenwich & Bridgeport

A human-gated pilot for extracting two test zoning variables from official PDFs:

- **L1** — Minimum Lot Size
- **U2** — Two-Family Permission

for **Greenwich, CT** and **Bridgeport, CT**.

This notebook is built as a sequence of **independent checkpoints**, not one automated
run. Each stage stops with a printed status before the next stage begins. There is no
scoring, subindex, or composite anywhere in this pilot — the deliverable is a small set of
verified, source-quoted facts per district, each backed by an exact quote and PDF page.


---

## Stage 0 — Setup and Scope

Defines the pilot's municipalities, variables, and guardrails, and checks the environment.
This stage does **not** open or parse any PDF, call any LLM, or create any schema/output
file beyond this notebook and the documentation files listed in `docs/` and `README.md`.


In [1]:
# --- Pilot configuration: municipalities ---
PILOT_MUNICIPALITIES = [
    {"name": "Greenwich", "state": "CT", "slug": "greenwich_ct"},
    {"name": "Bridgeport", "state": "CT", "slug": "bridgeport_ct"},
]

# --- Pilot configuration: variables under test ---
# Search-term lists are per-municipality: "default" applies unless a town has its own
# list. Bridgeport needed its own list after Stage 0/1 discovery showed it uses
# form-based vocabulary ("Double House A Building Type", "principal units") rather than
# "two-family"/"duplex" — see docs/U2_decision_tree.md and docs/L1_decision_tree.md for
# the full rationale. The reported variable label (L1 / U2) never changes — only the
# discovery vocabulary used to find candidate pages does.
PILOT_VARIABLES = {
    "L1": {
        "name": "Minimum Lot Size",
        "search_terms": {
            "default": [
                "minimum lot size", "minimum lot area", "minimum site area",
                "minimum zoning lot area", "lot area per dwelling unit",
                "dimensional standards", "bulk regulations",
                "schedule of required open spaces",
            ],
            "bridgeport_ct": [
                "minimum lot size", "minimum lot area", "lot width",
                "site coverage", "building type", "dimensional",
            ],
        },
    },
    "U2": {
        "name": "Two-Family Permission",
        "search_terms": {
            "default": [
                "two-family", "two family", "two-family dwelling", "duplex",
                "permitted uses", "use regulations", "special permit", "special exception",
            ],
            "bridgeport_ct": [
                "two-family", "two family", "duplex", "double house", "building type",
                "household living", "principal units", "attached house", "detached house",
                "neighborhood zone", "form-based",
            ],
        },
    },
}


def search_terms_for(variable_id: str, municipality_slug: str) -> list:
    """Resolve the effective search-term list for a variable + town, falling back to
    the variable's 'default' list when the town has no town-specific override."""
    terms = PILOT_VARIABLES[variable_id]["search_terms"]
    return terms.get(municipality_slug, terms["default"])


# District list is supplied by the user before Stage 2/3 need it. Left as an explicit
# placeholder here so Stage 0 can run and report its absence without blocking anything.
DISTRICTS = None

# --- Guardrail flags, asserted explicitly so they are visible, not just implied ---
SCORING_ENABLED = False
SUBINDEX_ENABLED = False
COMPOSITE_ENABLED = False
HUMAN_REVIEW_REQUIRED = True

print("Pilot configuration loaded.")
print(f"Municipalities: {len(PILOT_MUNICIPALITIES)}")
print(f"Variables: {len(PILOT_VARIABLES)}")


Pilot configuration loaded.
Municipalities: 2
Variables: 2


In [2]:
# --- Folder existence check (verify only — this cell never creates/moves/deletes anything) ---
from pathlib import Path

# Resolve relative to this notebook's own location, not whatever directory the editor
# happens to have open, so the check is portable across machines/IDEs.
PILOT_ROOT = Path.cwd()
if not (PILOT_ROOT / "data").exists() and (PILOT_ROOT.parent / "zoning_pdf_pilot" / "data").exists():
    PILOT_ROOT = PILOT_ROOT.parent / "zoning_pdf_pilot"

EXPECTED_FOLDERS = [
    "data/raw_pdfs/greenwich_ct",
    "data/raw_pdfs/bridgeport_ct",
    "data/source_manifest",
    "data/processed/extracted_text",
    "data/processed/source_packets",
    "data/processed/table_artifacts",
    "schemas",
    "outputs/manifests",
    "outputs/triage",
    "outputs/candidates",
    "outputs/review",
    "outputs/pilot_summary",
    "docs",
]

folder_check_rows = []
for rel in EXPECTED_FOLDERS:
    path = PILOT_ROOT / rel
    folder_check_rows.append({"Folder": rel, "Exists": path.exists()})

all_folders_ok = all(row["Exists"] for row in folder_check_rows)
for row in folder_check_rows:
    status = "OK" if row["Exists"] else "MISSING"
    print(f"  [{status}] {row['Folder']}")

print(f"\nPilot root resolved to: {PILOT_ROOT}")
print(f"All expected folders present: {all_folders_ok}")


  [OK] data/raw_pdfs/greenwich_ct
  [OK] data/raw_pdfs/bridgeport_ct
  [OK] data/source_manifest
  [OK] data/processed/extracted_text
  [OK] data/processed/source_packets
  [OK] data/processed/table_artifacts
  [OK] schemas
  [OK] outputs/manifests
  [OK] outputs/triage
  [OK] outputs/candidates
  [OK] outputs/review
  [OK] outputs/pilot_summary
  [OK] docs

Pilot root resolved to: /Users/sierramendoza/Documents/HAP-Repository/New_StageA/zoning_pdf_pilot
All expected folders present: True


In [3]:
# --- Raw-PDF presence check (list and count filenames only — no PDF is opened or parsed) ---
pdf_counts = {}
for muni in PILOT_MUNICIPALITIES:
    folder = PILOT_ROOT / "data" / "raw_pdfs" / muni["slug"]
    pdfs = sorted(folder.glob("*.pdf")) if folder.exists() else []
    pdf_counts[muni["name"]] = pdfs
    print(f"{muni['name']} ({muni['slug']}): {len(pdfs)} PDF(s) found")
    for p in pdfs:
        print(f"    - {p.name}")


Greenwich (greenwich_ct): 2 PDF(s) found
    - greenwich_ct.pdf
    - greenwich_zoning_map.pdf
Bridgeport (bridgeport_ct): 2 PDF(s) found
    - bridgeport_ct.pdf
    - bridgeport_zoning_map_2022.pdf


In [4]:
# --- Computed setup status ---
total_pdfs = sum(len(v) for v in pdf_counts.values())
any_muni_missing_pdfs = any(len(v) == 0 for v in pdf_counts.values())

if not all_folders_ok:
    pilot_status = "FOLDER_STRUCTURE_INCOMPLETE"
elif any_muni_missing_pdfs:
    pilot_status = "WAITING_FOR_DOCUMENT_UPLOAD"
else:
    pilot_status = "READY_FOR_STAGE_1"

print("ZONING PDF PILOT — SETUP CHECK")
print("=" * 59)
print(f"Municipalities: {len(PILOT_MUNICIPALITIES)}")
print(f"Variables: {len(PILOT_VARIABLES)}")
print()
print(f"Scoring enabled: {SCORING_ENABLED}")
print(f"Subindex enabled: {SUBINDEX_ENABLED}")
print(f"Composite enabled: {COMPOSITE_ENABLED}")
print(f"Human review required: {HUMAN_REVIEW_REQUIRED}")
print()
for muni in PILOT_MUNICIPALITIES:
    print(f"{muni['name']} PDFs found: {len(pdf_counts[muni['name']])}")
print()
print(f"District list supplied: {DISTRICTS is not None}")
print()
print(f"Pilot status: {pilot_status}")


ZONING PDF PILOT — SETUP CHECK
Municipalities: 2
Variables: 2

Scoring enabled: False
Subindex enabled: False
Composite enabled: False
Human review required: True

Greenwich PDFs found: 2
Bridgeport PDFs found: 2

District list supplied: False

Pilot status: READY_FOR_STAGE_1


---

## Stage 1 — Register Uploaded Official PDFs

This stage builds a **source inventory**, nothing more. It proves the four manually
uploaded PDFs were received, can be opened, and have stable identity fingerprints (hash,
page count, basic PDF metadata, a short first-page preview).

It does **not** prove the documents are current, complete, legally governing, or relevant
to either pilot variable (L1 or U2) — that comes later. No page content beyond a short
first-page identity preview is read in this stage, no search terms are applied, and no
LLM is called.


In [5]:
# --- Stage 1: manifest registration utilities ---
import hashlib
from datetime import datetime, timezone

import pdfplumber

NEEDS_MANUAL = "needs_manual_metadata_entry"
PREVIEW_MAX_CHARS = 300
PDF_METADATA_FIELDS = (
    "pdf_metadata_title", "pdf_metadata_author", "pdf_metadata_creator",
    "pdf_metadata_producer", "pdf_metadata_creation_date", "pdf_metadata_modification_date",
)


def sha256_of_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def infer_document_role(filename: str) -> str:
    """Transparent, filename-only role inference. No file content is used."""
    lower = filename.lower()
    if "map" in lower:
        return "zoning_map"
    if any(kw in lower for kw in ("zoning", "zone", "building_zone", "regulations", "code")):
        return "zoning_code"
    return "unknown"


def register_pdf(path: Path, municipality: dict) -> dict:
    """Read only identity-level facts about one PDF: hash, page count, PDF metadata,
    and a short first-page text preview. Never reads beyond page 1's text content.
    """
    raw_bytes = path.read_bytes()
    raw_hash = sha256_of_bytes(raw_bytes)
    role = infer_document_role(path.name)
    registered_at = datetime.now(timezone.utc).isoformat()

    record = {
        "source_id": f"{municipality['slug']}__{role}__{raw_hash[:12]}",
        "municipality": municipality["name"],
        "state": municipality["state"],
        "municipality_slug": municipality["slug"],
        "document_role": role,
        "local_filename": path.name,
        "relative_path": str(path.relative_to(PILOT_ROOT)),
        "file_size_bytes": len(raw_bytes),
        "pdf_page_count": None,
        "pdf_is_readable": False,
        "raw_pdf_sha256": raw_hash,
        "first_page_text_preview": None,
        "first_page_text_sha256": None,
        "pdf_metadata_title": NEEDS_MANUAL,
        "pdf_metadata_author": NEEDS_MANUAL,
        "pdf_metadata_creator": NEEDS_MANUAL,
        "pdf_metadata_producer": NEEDS_MANUAL,
        "pdf_metadata_creation_date": NEEDS_MANUAL,
        "pdf_metadata_modification_date": NEEDS_MANUAL,
        # Provenance facts that cannot be determined from the file itself — manual entry
        # required later; never guessed here.
        "source_url": NEEDS_MANUAL,
        "official_document_title": NEEDS_MANUAL,
        "adoption_date": NEEDS_MANUAL,
        "effective_date": NEEDS_MANUAL,
        "current_through_date": NEEDS_MANUAL,
        "authority_status": NEEDS_MANUAL,
        "source_notes": NEEDS_MANUAL,
        "metadata_status": "registration_failed",
        "registered_at_utc": registered_at,
        "registration_status": "failed",
        "registration_error": None,
    }

    try:
        with pdfplumber.open(path) as pdf:
            record["pdf_page_count"] = len(pdf.pages)
            record["pdf_is_readable"] = True

            meta = pdf.metadata or {}
            record["pdf_metadata_title"] = meta.get("Title") or NEEDS_MANUAL
            record["pdf_metadata_author"] = meta.get("Author") or NEEDS_MANUAL
            record["pdf_metadata_creator"] = meta.get("Creator") or NEEDS_MANUAL
            record["pdf_metadata_producer"] = meta.get("Producer") or NEEDS_MANUAL
            record["pdf_metadata_creation_date"] = meta.get("CreationDate") or NEEDS_MANUAL
            record["pdf_metadata_modification_date"] = meta.get("ModDate") or NEEDS_MANUAL

            first_page_text = (pdf.pages[0].extract_text() or "") if pdf.pages else ""
            preview = first_page_text[:PREVIEW_MAX_CHARS].strip()
            record["first_page_text_preview"] = preview if preview else NEEDS_MANUAL
            record["first_page_text_sha256"] = sha256_of_bytes(first_page_text.encode("utf-8"))

        present = [record[k] != NEEDS_MANUAL for k in PDF_METADATA_FIELDS]
        if all(present):
            record["metadata_status"] = "registered_metadata_complete"
        elif any(present):
            record["metadata_status"] = "registered_metadata_partially_available"
        else:
            record["metadata_status"] = "registered_needs_manual_metadata"
        record["registration_status"] = "success"
    except Exception as exc:
        record["registration_error"] = str(exc)
        record["metadata_status"] = "registration_failed"
        record["registration_status"] = "failed"

    return record


print("Stage 1 registration utilities defined.")


Stage 1 registration utilities defined.


In [6]:
# --- Run registration over every PDF discovered in the raw_pdfs folders ---
import pandas as pd

manifest_records = []
for muni in PILOT_MUNICIPALITIES:
    folder = PILOT_ROOT / "data" / "raw_pdfs" / muni["slug"]
    for pdf_path in sorted(folder.glob("*.pdf")):
        manifest_records.append(register_pdf(pdf_path, muni))

manifest_df = pd.DataFrame(manifest_records)
display_cols = ["source_id", "municipality", "document_role", "local_filename",
                "pdf_page_count", "pdf_is_readable", "raw_pdf_sha256", "metadata_status",
                "registration_status"]
manifest_df[display_cols]


,source_id,municipality,document_role,local_filename,pdf_page_count,pdf_is_readable,raw_pdf_sha256,metadata_status,registration_status
0,greenwich_ct__unknown__18417c5e9074,Greenwich,unknown,greenwich_ct.pdf,313,True,18417c5e9074d765ab4483571b9ac3d6aeb162b37b1b30...,registered_metadata_partially_available,success
1,greenwich_ct__zoning_map__b52d3042d0a1,Greenwich,zoning_map,greenwich_zoning_map.pdf,1,True,b52d3042d0a15a8000cb90f564ac6901b2d97e37ea5362...,registered_metadata_partially_available,success
2,bridgeport_ct__unknown__afe7c678fa5d,Bridgeport,unknown,bridgeport_ct.pdf,304,True,afe7c678fa5dfc8581cca968c59eac168aa3b69b0807d2...,registered_metadata_partially_available,success
3,bridgeport_ct__zoning_map__2db15f1c1a52,Bridgeport,zoning_map,bridgeport_zoning_map_2022.pdf,1,True,2db15f1c1a520d51903d6dac7a864cddda434ff93bcfec...,registered_metadata_partially_available,success


In [7]:
# --- Save manifest (CSV + JSON) and print the Stage 1 summary ---
manifest_csv_path = PILOT_ROOT / "data" / "source_manifest" / "source_manifest_v1.csv"
manifest_json_path = PILOT_ROOT / "data" / "source_manifest" / "source_manifest_v1.json"

manifest_df.to_csv(manifest_csv_path, index=False)
manifest_df.to_json(manifest_json_path, orient="records", indent=2)

registered_count = len(manifest_records)
readable_count = sum(1 for r in manifest_records if r["pdf_is_readable"])
failed_count = sum(1 for r in manifest_records if r["registration_status"] == "failed")

print("=" * 72)
print("ZONING PDF PILOT — STAGE 1: SOURCE REGISTRATION")
print("=" * 72)
print()
print(f"Registered PDFs: {registered_count}")
print(f"Readable PDFs: {readable_count}")
print(f"Registration failures: {failed_count}")
print()

for muni in PILOT_MUNICIPALITIES:
    print(f"{muni['name']}, {muni['state']}:")
    for r in [r for r in manifest_records if r["municipality"] == muni["name"]]:
        print(f"  - {r['local_filename']}")
        print(f"    Role: {r['document_role']}")
        print(f"    Pages: {r['pdf_page_count']}")
        print(f"    SHA-256: {r['raw_pdf_sha256'][:16]}...")
        print(f"    Metadata status: {r['metadata_status']}")
        if r["registration_error"]:
            print(f"    ERROR: {r['registration_error']}")
        print()

print(f"Manifest CSV: {manifest_csv_path.relative_to(PILOT_ROOT)}")
print(f"Manifest JSON: {manifest_json_path.relative_to(PILOT_ROOT)}")
print()

if failed_count == 0 and readable_count == registered_count and registered_count > 0:
    stage1_result = "PASS"
elif readable_count > 0:
    stage1_result = "PARTIAL"
else:
    stage1_result = "FAIL"

print(f"STAGE 1 RESULT: {stage1_result}")
print("Meaning: Uploaded PDF files were registered, opened successfully,")
print("and fingerprinted. This does not yet establish currentness,")
print("legal authority, district applicability, or variable-level evidence.")
print("=" * 72)


ZONING PDF PILOT — STAGE 1: SOURCE REGISTRATION

Registered PDFs: 4
Readable PDFs: 4
Registration failures: 0

Greenwich, CT:
  - greenwich_ct.pdf
    Role: unknown
    Pages: 313
    SHA-256: 18417c5e9074d765...
    Metadata status: registered_metadata_partially_available

  - greenwich_zoning_map.pdf
    Role: zoning_map
    Pages: 1
    SHA-256: b52d3042d0a15a80...
    Metadata status: registered_metadata_partially_available

Bridgeport, CT:
  - bridgeport_ct.pdf
    Role: unknown
    Pages: 304
    SHA-256: afe7c678fa5dfc85...
    Metadata status: registered_metadata_partially_available

  - bridgeport_zoning_map_2022.pdf
    Role: zoning_map
    Pages: 1
    SHA-256: 2db15f1c1a520d51...
    Metadata status: registered_metadata_partially_available

Manifest CSV: data/source_manifest/source_manifest_v1.csv
Manifest JSON: data/source_manifest/source_manifest_v1.json

STAGE 1 RESULT: PASS
Meaning: Uploaded PDF files were registered, opened successfully,
and fingerprinted. This does no

---

## Stage 2 — Create Source Packets

This stage reads the two main zoning-code PDFs (map PDFs are excluded — a map page has
no searchable zoning text) and finds **candidate pages** for L1 and U2, using the
per-municipality search-term lists defined in Stage 0.

A source packet is **labeled raw material**, not an answer: it records which page(s)
matched which term, with a few pages of surrounding context preserved (a table's header
or a definition it depends on can sit on an adjacent page), plus the literal extracted
text. No relevance judgment, triage classification, or LLM call happens in this stage.

This stage resolves `document_role` through the Stage 1 manifest **and** the manual
`role_overrides_v1.csv` file created after Stage 1 review — `greenwich_ct.pdf` and
`bridgeport_ct.pdf` were registered as `unknown` by the filename-only rule, and the
override file is what tells this stage they are the two towns' zoning codes.


In [8]:
# --- Resolve effective document role: Stage 1 manifest + manual role overrides ---
manifest_for_stage2 = pd.read_csv(PILOT_ROOT / "data" / "source_manifest" / "source_manifest_v1.csv")

overrides_path = PILOT_ROOT / "data" / "source_manifest" / "role_overrides_v1.csv"
overrides_df = pd.read_csv(overrides_path) if overrides_path.exists() else pd.DataFrame(
    columns=["local_filename", "raw_pdf_sha256", "reviewed_role"]
)

override_by_hash = dict(zip(overrides_df["raw_pdf_sha256"], overrides_df["reviewed_role"]))

def effective_role(row) -> str:
    """Reviewed role if a human has recorded one for this exact file hash, else the
    Stage 1 filename-inferred role. Two separate, traceable ideas, never merged in place."""
    return override_by_hash.get(row["raw_pdf_sha256"], row["document_role"])

manifest_for_stage2["effective_role"] = manifest_for_stage2.apply(effective_role, axis=1)
manifest_for_stage2[["local_filename", "municipality_slug", "document_role", "effective_role"]]


,local_filename,municipality_slug,document_role,effective_role
0,greenwich_ct.pdf,greenwich_ct,unknown,zoning_code
1,greenwich_zoning_map.pdf,greenwich_ct,zoning_map,zoning_map
2,bridgeport_ct.pdf,bridgeport_ct,unknown,zoning_code
3,bridgeport_zoning_map_2022.pdf,bridgeport_ct,zoning_map,zoning_map


In [9]:
# --- Stage 2: source-packet creation utilities ---
import re

CONTEXT_PAGES = 1  # pages of surrounding context to pull in on each side of a hit


def extract_all_page_texts(pdf_path: Path) -> list:
    """Return a list of page texts (1-indexed position = page number). Reads the whole
    document once per PDF so later steps don't re-open it per search term."""
    with pdfplumber.open(pdf_path) as pdf:
        return [(page.extract_text() or "") for page in pdf.pages]


def find_term_hit_pages(page_texts: list, term: str) -> list:
    """Return the 1-indexed page numbers where `term` appears (case-insensitive)."""
    lowered_term = term.lower()
    return [i + 1 for i, text in enumerate(page_texts) if lowered_term in text.lower()]


def group_consecutive(pages: list) -> list:
    """Group consecutive page numbers into ranges, e.g. [5,6,7,20] -> [(5,7), (20,20)]."""
    if not pages:
        return []
    pages = sorted(set(pages))
    groups, start, prev = [], pages[0], pages[0]
    for p in pages[1:]:
        if p == prev + 1:
            prev = p
            continue
        groups.append((start, prev))
        start, prev = p, p
    groups.append((start, prev))
    return groups


def build_packet(municipality: dict, variable_id: str, term: str, page_range: tuple,
                  page_texts: list, source_filename: str, packet_seq: int) -> dict:
    """Build one source packet covering a hit page range plus CONTEXT_PAGES of padding
    on each side, so a table header or definition on an adjacent page isn't lost."""
    start, end = page_range
    ctx_start = max(1, start - CONTEXT_PAGES)
    ctx_end = min(len(page_texts), end + CONTEXT_PAGES)

    combined_text = "\n\n".join(page_texts[ctx_start - 1:ctx_end])
    combined_text = re.sub(r"[ \t]+", " ", combined_text).strip()

    return {
        "packet_id": f"{municipality['slug']}_{variable_id}_{packet_seq:03d}",
        "municipality": municipality["name"],
        "variable_target": variable_id,
        "source_filename": source_filename,
        "matched_pages": f"{start}-{end}" if start != end else str(start),
        "context_pages": f"{ctx_start}-{ctx_end}",
        "search_term_matched": term,
        "extracted_text": combined_text,
        "text_length": len(combined_text),
        "preview": combined_text[:200],
    }


print("Stage 2 packet-creation utilities defined.")


Stage 2 packet-creation utilities defined.


In [10]:
# --- Run packet creation: zoning_code PDFs only, per-municipality term lists ---
packet_records = []

for muni in PILOT_MUNICIPALITIES:
    muni_rows = manifest_for_stage2[manifest_for_stage2["municipality_slug"] == muni["slug"]]
    code_rows = muni_rows[muni_rows["effective_role"] == "zoning_code"]

    if code_rows.empty:
        print(f"[SKIP] {muni['name']}: no PDF has effective_role == 'zoning_code' — check role_overrides_v1.csv")
        continue

    for _, code_row in code_rows.iterrows():
        pdf_path = PILOT_ROOT / code_row["relative_path"]
        print(f"Reading {code_row['local_filename']} ({code_row['pdf_page_count']} pages)...")
        page_texts = extract_all_page_texts(pdf_path)

        for variable_id in PILOT_VARIABLES:
            terms = search_terms_for(variable_id, muni["slug"])
            seq = 0
            for term in terms:
                hit_pages = find_term_hit_pages(page_texts, term)
                for page_range in group_consecutive(hit_pages):
                    seq += 1
                    packet_records.append(build_packet(
                        muni, variable_id, term, page_range, page_texts,
                        code_row["local_filename"], seq,
                    ))

packets_df = pd.DataFrame(packet_records)
print(f"\nTotal packets created: {len(packets_df)}")
packets_df[["packet_id", "municipality", "variable_target", "matched_pages",
            "search_term_matched", "text_length"]]


Reading greenwich_ct.pdf (313 pages)...


Reading bridgeport_ct.pdf (304 pages)...



Total packets created: 314


,packet_id,municipality,variable_target,matched_pages,search_term_matched,text_length
0,greenwich_ct_L1_001,Greenwich,L1,48,minimum lot size,8034
1,greenwich_ct_L1_002,Greenwich,L1,60-61,minimum lot size,10072
2,greenwich_ct_L1_003,Greenwich,L1,106,minimum lot size,10963
3,greenwich_ct_L1_004,Greenwich,L1,110,minimum lot size,10252
4,greenwich_ct_L1_005,Greenwich,L1,115,minimum lot size,12193
...,...,...,...,...,...,...
309,bridgeport_ct_U2_070,Bridgeport,U2,134,detached house,7829
310,bridgeport_ct_U2_071,Bridgeport,U2,144,detached house,13991
311,bridgeport_ct_U2_072,Bridgeport,U2,9-11,neighborhood zone,10313
312,bridgeport_ct_U2_073,Bridgeport,U2,13,neighborhood zone,6355


In [11]:
# --- Save packet inventory (CSV + JSON) and print the Stage 2 summary ---
packets_csv_path = PILOT_ROOT / "data" / "processed" / "source_packets" / "source_packets_v1.csv"
packets_json_path = PILOT_ROOT / "data" / "processed" / "source_packets" / "source_packets_v1.json"

packets_df.to_csv(packets_csv_path, index=False)
packets_df.to_json(packets_json_path, orient="records", indent=2)

print("=" * 72)
print("ZONING PDF PILOT — STAGE 2: SOURCE PACKET CREATION")
print("=" * 72)
print()
for muni in PILOT_MUNICIPALITIES:
    muni_packets = packets_df[packets_df["municipality"] == muni["name"]] if not packets_df.empty else packets_df
    print(f"{muni['name']}, {muni['state']}:")
    for variable_id, variable in PILOT_VARIABLES.items():
        count = len(muni_packets[muni_packets["variable_target"] == variable_id]) if not muni_packets.empty else 0
        terms_used = search_terms_for(variable_id, muni["slug"])
        print(f"  {variable_id} ({variable['name']}): {count} packet(s) "
              f"— {len(terms_used)} search term(s) tried")
    print()

print(f"Total packets: {len(packets_df)}")
print(f"Packet CSV: {packets_csv_path.relative_to(PILOT_ROOT)}")
print(f"Packet JSON: {packets_json_path.relative_to(PILOT_ROOT)}")
print()

stage2_result = "PASS" if len(packets_df) > 0 else "FAIL"
print(f"STAGE 2 RESULT: {stage2_result}")
print("Meaning: Candidate pages were located and preserved with surrounding context.")
print("No packet has been classified as relevant, triaged, or interpreted by an LLM.")
print("Every packet remains labeled raw material requiring Stage 3 triage.")
print("=" * 72)


ZONING PDF PILOT — STAGE 2: SOURCE PACKET CREATION

Greenwich, CT:
  L1 (Minimum Lot Size): 23 packet(s) — 8 search term(s) tried
  U2 (Two-Family Permission): 138 packet(s) — 8 search term(s) tried

Bridgeport, CT:
  L1 (Minimum Lot Size): 79 packet(s) — 6 search term(s) tried
  U2 (Two-Family Permission): 74 packet(s) — 11 search term(s) tried

Total packets: 314
Packet CSV: data/processed/source_packets/source_packets_v1.csv
Packet JSON: data/processed/source_packets/source_packets_v1.json

STAGE 2 RESULT: PASS
Meaning: Candidate pages were located and preserved with surrounding context.
No packet has been classified as relevant, triaged, or interpreted by an LLM.
Every packet remains labeled raw material requiring Stage 3 triage.


---

## Stage 3A — Groq Source-Packet Triage

Only a small, deliberately selected batch of the 314 Stage 2 packets is classified here
(≤20 total). The output is **candidate triage only** — a judgment about whether a packet
*looks* worth sending to a later human-reviewed extraction step.

This stage does **not** produce a legal conclusion, a final L1/U2 value, a fact sheet, a
score, or a substitute for human verification. Every triage result requires later human
review before anything moves to Stage 3B extraction.

The Groq API key is loaded locally to authenticate the request — it is never printed,
logged, written to notebook output, or committed. This cell only ever prints
`Groq key present: True/False`.


In [12]:
# --- Load Stage 2 packets and select a bounded, deterministic triage batch (<=20) ---
packets_for_triage = pd.read_json(PILOT_ROOT / "data" / "processed" / "source_packets" / "source_packets_v1.json")
print(f"Loaded {len(packets_for_triage)} packets from Stage 2.")

# Vocabulary anchors that must be represented in the batch, so the pilot's known
# discovery findings (form-based Bridgeport vocabulary, Greenwich dimensional/use
# language) are deliberately tested rather than left to chance.
REQUIRED_ANCHOR_TERMS = {
    ("Greenwich", "L1"): ["minimum lot size", "minimum lot area", "schedule of required open spaces", "height and bulk"],
    ("Greenwich", "U2"): ["two-family", "two family", "permitted uses", "use regulations"],
    ("Bridgeport", "L1"): ["minimum lot size", "minimum lot area", "lot width", "site coverage"],
    ("Bridgeport", "U2"): ["double house", "building type", "principal units", "household living"],
}
MAX_PER_BUCKET = 5


def score_packet(row) -> tuple:
    """Lower score = preferred. Favors a specific (non-broad-procedural) search term and
    a short, tight page range over long/broad matches, as a proxy for relevance density."""
    broad_terms = {"permitted uses", "use regulations", "special permit", "special exception",
                   "dimensional", "building type"}
    is_broad = row["search_term_matched"] in broad_terms
    try:
        start, end = (row["matched_pages"].split("-") + [row["matched_pages"]])[:2]
        page_span = abs(int(end) - int(start)) if "-" in row["matched_pages"] else 0
    except Exception:
        page_span = 99
    return (1 if is_broad else 0, page_span, row["text_length"])


selected_ids = []
selection_log = []

for (muni_name, var_id), anchor_terms in REQUIRED_ANCHOR_TERMS.items():
    bucket = packets_for_triage[
        (packets_for_triage["municipality"] == muni_name) & (packets_for_triage["variable_target"] == var_id)
    ].copy()

    chosen = []

    # Step 1: guarantee at least one packet per required anchor term, if available —
    # picking the tightest/most-specific match for that term.
    for term in anchor_terms:
        if len(chosen) >= MAX_PER_BUCKET:
            break
        term_matches = bucket[bucket["search_term_matched"] == term]
        term_matches = term_matches[~term_matches["packet_id"].isin(chosen)]
        if not term_matches.empty:
            scored = term_matches.assign(_score=term_matches.apply(score_packet, axis=1))
            best_row = scored.sort_values("_score").iloc[0]
            chosen.append(best_row["packet_id"])
            selection_log.append((best_row["packet_id"], f"anchor term '{term}'"))

    # Step 2: fill remaining slots with the lowest-scoring (most specific/least
    # redundant-looking) remaining packets in this bucket.
    remaining = bucket[~bucket["packet_id"].isin(chosen)].copy()
    remaining["_score"] = remaining.apply(score_packet, axis=1)
    remaining = remaining.sort_values("_score")
    for _, row in remaining.iterrows():
        if len(chosen) >= MAX_PER_BUCKET:
            break
        chosen.append(row["packet_id"])
        selection_log.append((row["packet_id"], "lowest-redundancy fill"))

    selected_ids.extend(chosen)

print(f"\nSelected {len(selected_ids)} packets total (cap: 20).\n")
print("Selection method: for each (municipality, variable) bucket, first guarantee one")
print("packet per required anchor term (picking the tightest/most-specific match for")
print("that term), then fill remaining slots up to 5 with the lowest-scoring remaining")
print("packets, where 'broad procedural' search terms and longer page spans score worse.\n")
for pid, reason in selection_log:
    print(f"  {pid}: {reason}")


Loaded 314 packets from Stage 2.

Selected 20 packets total (cap: 20).

Selection method: for each (municipality, variable) bucket, first guarantee one
packet per required anchor term (picking the tightest/most-specific match for
that term), then fill remaining slots up to 5 with the lowest-scoring remaining
packets, where 'broad procedural' search terms and longer page spans score worse.

  greenwich_ct_L1_008: anchor term 'minimum lot size'
  greenwich_ct_L1_010: anchor term 'minimum lot area'
  greenwich_ct_L1_016: anchor term 'schedule of required open spaces'
  greenwich_ct_L1_018: lowest-redundancy fill
  greenwich_ct_L1_021: lowest-redundancy fill
  greenwich_ct_U2_001: anchor term 'two-family'
  greenwich_ct_U2_014: anchor term 'two family'
  greenwich_ct_U2_022: anchor term 'permitted uses'
  greenwich_ct_U2_056: anchor term 'use regulations'
  greenwich_ct_U2_019: lowest-redundancy fill
  bridgeport_ct_L1_003: anchor term 'minimum lot area'
  bridgeport_ct_L1_013: anchor term

In [13]:
# --- Groq key presence (safe) + documented model selection ---
import os
from datetime import datetime, timezone

from dotenv import load_dotenv

load_dotenv(PILOT_ROOT.parent / ".env")
groq_api_key = os.environ.get("GROQ_API_KEY")
print(f"Groq key present: {bool(groq_api_key)}")

CHOSEN_MODEL_ID = None
model_selection_timestamp = None

if groq_api_key:
    from groq import Groq

    groq_client = Groq(api_key=groq_api_key)
    available_models = groq_client.models.list().data  # metadata only, no content sent

    def is_eligible(m) -> bool:
        feats = getattr(m, "supported_features", None) or []
        in_mod = getattr(m, "input_modalities", None) or []
        out_mod = getattr(m, "output_modalities", None) or []
        return (
            m.active
            and in_mod == ["text"]
            and out_mod == ["text"]
            and "json_mode" in feats                   # required for the triage schema
            and "guard" not in m.id.lower()             # exclude prompt-guard classifiers
            and "safeguard" not in m.id.lower()         # exclude safety classifiers
            and "preview" not in m.id.lower()           # prefer stable over preview
        )

    def prompt_price(m) -> float:
        pricing = getattr(m, "pricing", None) or {}
        try:
            return float(pricing.get("prompt", "inf"))
        except Exception:
            return float("inf")

    eligible = [m for m in available_models if is_eligible(m)]
    # Prefer structured_outputs support (strongest JSON enforcement), then the smallest
    # context window that's still adequate, then the cheapest prompt price as a final,
    # deterministic tiebreaker — the account's model-listing order is not guaranteed
    # stable across calls, so it is never used as a tiebreaker.
    eligible.sort(key=lambda m: (
        0 if "structured_outputs" in (getattr(m, "supported_features", None) or []) else 1,
        getattr(m, "context_window", 0),
        prompt_price(m),
    ))

    if eligible:
        CHOSEN_MODEL_ID = eligible[0].id
        model_selection_timestamp = datetime.now(timezone.utc).isoformat()
        print(f"\nChosen model: {CHOSEN_MODEL_ID}")
        print("Selection rule: active on this account; text-in/text-out only (excludes audio/")
        print("image/embedding-only models); supports json_mode (required for the triage")
        print("schema); excludes safety/prompt-guard classifiers and preview models; prefers")
        print("structured_outputs support; ties broken by smallest adequate context window,")
        print("then lowest prompt price — favors cost/latency for a short classification task")
        print("over a larger reasoning-oriented model.")
        print(f"Selected at: {model_selection_timestamp}")
    else:
        print("\nNo eligible model found. Stage 3A cannot proceed.")


Groq key present: True



Chosen model: openai/gpt-oss-20b
Selection rule: active on this account; text-in/text-out only (excludes audio/
image/embedding-only models); supports json_mode (required for the triage
schema); excludes safety/prompt-guard classifiers and preview models; prefers
structured_outputs support; ties broken by smallest adequate context window,
then lowest prompt price — favors cost/latency for a short classification task
over a larger reasoning-oriented model.
Selected at: 2026-10-02T17:47:16.269321+00:00


In [14]:
# --- Triage prompt, JSON schema, and deterministic Python-side validation ---
import json
import re as re_triage

PROMPT_VERSION = "triage_v1"

TRIAGE_CATEGORIES = [
    "DIMENSIONAL_TABLE_OR_SCHEDULE", "GOVERNING_NUMERIC_STANDARD",
    "USE_TABLE_OR_PERMISSION_RULE", "GOVERNING_PERMISSION_RULE", "DEFINITION_ONLY",
    "PROCEDURAL_OR_APPROVAL_RULE", "FEE_OR_FINANCIAL_RULE", "CROSS_REFERENCE_ONLY",
    "OVERLAY_OR_SPECIAL_DISTRICT_RULE", "NONCONFORMING_OR_EXISTING_LOT_EXCEPTION",
    "REMOVED_RESERVED_OR_EXTERNAL_SOURCE_NOTICE", "IRRELEVANT", "UNRESOLVED",
]

TRIAGE_INSTRUCTIONS = """You are triaging a public municipal zoning source packet.

Your task is not to make a legal conclusion or fill a final fact sheet.
Classify only whether this packet appears relevant for later human-reviewed
extraction of the specified variable.

Use only the supplied text. Do not use outside knowledge. Do not infer
missing information. If uncertain, return UNRESOLVED and
send_to_extraction = false.

For L1, do not treat a reference to lot size in a fee, tax, wetlands,
application, filing, subdivision-administration, or procedural context as
a governing minimum lot-size rule.

For U2, do not treat an ADU provision as two-family permission. Do not
treat a definition alone as proof that two-family housing is permitted.
For form-based codes, a building-type rule may be relevant only if the
packet itself indicates the type can contain two principal dwelling units
or otherwise supports a two-unit interpretation; if unclear, return
UNRESOLVED.

A district name, building type, or defined term is not by itself proof
of housing permission.

Return null rather than inventing a field."""

REQUIRED_TRIAGE_FIELDS = {
    "packet_id": str, "triage_class": str, "target_variable_relevance": str,
    "likely_governing_rule": bool, "send_to_extraction": bool,
    "districts_or_building_types_mentioned": list, "numeric_standard_present": bool,
    "permission_pathway_present": bool, "table_or_schedule_present": bool,
    "footnote_or_exception_present": bool, "cross_reference_present": bool,
    "short_reason": str, "follow_up_needed": bool, "follow_up_reason": (str, type(None)),
    "source_quote_for_triage": (str, type(None)), "triage_confidence": (int, float),
}


def build_triage_prompt(packet_row, variable_name: str) -> str:
    """Only the documented, non-secret fields go into the prompt — never the map, never
    unrelated packets, never any credential."""
    return f"""{TRIAGE_INSTRUCTIONS}

PACKET METADATA:
municipality: {packet_row['municipality']}
state: CT
target_variable_id: {packet_row['variable_target']}
target_variable_name: {variable_name}
search_term_matched: {packet_row['search_term_matched']}
source_document_filename: {packet_row['source_filename']}
page_range: {packet_row['matched_pages']}

PACKET TEXT:
{packet_row['extracted_text']}

Respond with ONLY a single JSON object with exactly these fields:
{{
  "packet_id": "{packet_row['packet_id']}",
  "triage_class": one of {TRIAGE_CATEGORIES},
  "target_variable_relevance": "direct | indirect | none | unresolved",
  "likely_governing_rule": true/false,
  "send_to_extraction": true/false,
  "districts_or_building_types_mentioned": ["string", ...],
  "numeric_standard_present": true/false,
  "permission_pathway_present": true/false,
  "table_or_schedule_present": true/false,
  "footnote_or_exception_present": true/false,
  "cross_reference_present": true/false,
  "short_reason": "max two sentences, based only on this packet's text",
  "follow_up_needed": true/false,
  "follow_up_reason": "string or null",
  "source_quote_for_triage": "short exact quote from the packet text, or null",
  "triage_confidence": 0.0 to 1.0
}}"""


def normalize_whitespace(text: str) -> str:
    return re_triage.sub(r"\s+", " ", text or "").strip().lower()


def validate_triage_response(raw_text: str, expected_packet_id: str, packet_text: str) -> dict:
    """Deterministic Python-side validation. Returns {"status": ..., "parsed": ..., "issues": [...]}."""
    issues = []
    try:
        parsed = json.loads(raw_text)
    except json.JSONDecodeError as e:
        return {"status": "invalid_response_needs_retry", "parsed": None, "issues": [f"not_valid_json: {e}"]}

    if not isinstance(parsed, dict):
        return {"status": "invalid_response_needs_retry", "parsed": None, "issues": ["response_not_a_json_object"]}

    extra_fields = set(parsed.keys()) - set(REQUIRED_TRIAGE_FIELDS.keys())
    known = {k: v for k, v in parsed.items() if k in REQUIRED_TRIAGE_FIELDS}
    raw_extra = {k: parsed[k] for k in extra_fields}

    if parsed.get("packet_id") != expected_packet_id:
        issues.append(f"packet_id_mismatch: got {parsed.get('packet_id')!r}, expected {expected_packet_id!r}")

    if parsed.get("triage_class") not in TRIAGE_CATEGORIES:
        issues.append(f"triage_class_not_in_allowed_set: {parsed.get('triage_class')!r}")

    for field, expected_type in REQUIRED_TRIAGE_FIELDS.items():
        if field not in parsed:
            issues.append(f"missing_field: {field}")
            continue
        if not isinstance(parsed[field], expected_type):
            issues.append(f"wrong_type: {field} is {type(parsed[field]).__name__}, expected {expected_type}")

    confidence = parsed.get("triage_confidence")
    if isinstance(confidence, (int, float)) and not (0.0 <= confidence <= 1.0):
        issues.append(f"confidence_out_of_range: {confidence}")

    quote = parsed.get("source_quote_for_triage")
    if quote and normalize_whitespace(quote) not in normalize_whitespace(packet_text):
        issues.append("source_quote_not_found_in_packet_text")

    reason = parsed.get("short_reason", "")
    if not reason or not reason.strip():
        issues.append("short_reason_empty")
    elif reason.count(".") > 2 and len(reason) > 400:
        issues.append("short_reason_too_long")

    status = "completed" if not issues else "invalid_response_needs_retry"
    return {"status": status, "parsed": known, "issues": issues, "raw_extra_fields": raw_extra}


print(f"Triage schema, prompt builder, and validator defined (prompt version: {PROMPT_VERSION}).")


Triage schema, prompt builder, and validator defined (prompt version: triage_v1).


In [15]:
# --- Run triage on the selected batch: one request per packet, strict error handling ---
run_started_at = datetime.now(timezone.utc).isoformat()
triage_results = []
attempted = 0
provider_error_count = 0
stage3a_stopped_early = False

selected_batch = packets_for_triage[packets_for_triage["packet_id"].isin(selected_ids)]

if CHOSEN_MODEL_ID is None:
    print("No model selected — skipping triage run.")
else:
    for _, packet_row in selected_batch.iterrows():
        attempted += 1
        variable_name = PILOT_VARIABLES[packet_row["variable_target"]]["name"]
        prompt = build_triage_prompt(packet_row, variable_name)
        request_ts = datetime.now(timezone.utc).isoformat()

        record = {
            "packet_id": packet_row["packet_id"], "municipality": packet_row["municipality"],
            "state": "CT", "variable_id": packet_row["variable_target"],
            "source_document_filename": packet_row["source_filename"],
            "page_range": packet_row["matched_pages"],
            "matched_search_term": packet_row["search_term_matched"],
            "model_provider": "groq", "model_id": CHOSEN_MODEL_ID,
            "prompt_version": PROMPT_VERSION, "request_timestamp_utc": request_ts,
            "raw_model_response": None, "parsed_triage_response": None,
            "validation_status": None, "validation_issues": None,
        }

        try:
            response = groq_client.chat.completions.create(
                model=CHOSEN_MODEL_ID,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
            )
            raw_text = response.choices[0].message.content
            record["raw_model_response"] = raw_text

            validation = validate_triage_response(raw_text, packet_row["packet_id"], packet_row["extracted_text"])
            record["parsed_triage_response"] = validation["parsed"]
            record["validation_status"] = validation["status"]
            record["validation_issues"] = validation["issues"]

        except Exception as exc:
            provider_error_count += 1
            record["validation_status"] = "provider_error"
            record["validation_issues"] = [f"{type(exc).__name__}: {str(exc)[:400]}"]

        triage_results.append(record)

        if attempted >= 4 and (provider_error_count / attempted) >= 0.25:
            stage3a_stopped_early = True
            print(f"STAGE_3A_STATUS: PROVIDER_UNSTABLE — stopping after packet {attempted}/{len(selected_batch)}.")
            break

run_finished_at = datetime.now(timezone.utc).isoformat()
triage_results_df = pd.DataFrame(triage_results)
print(f"Attempted: {attempted}, provider errors: {provider_error_count}")
if not triage_results_df.empty:
    triage_results_df[["packet_id", "municipality", "variable_id", "validation_status"]]


STAGE_3A_STATUS: PROVIDER_UNSTABLE — stopping after packet 8/20.
Attempted: 8, provider errors: 2


In [16]:
# --- Save Stage 3A outputs (never overwrite an existing batch) and print the summary ---
triage_dir = PILOT_ROOT / "outputs" / "triage"
csv_path = triage_dir / "triage_batch_v1.csv"
json_path = triage_dir / "triage_batch_v1.json"
summary_path = triage_dir / "triage_batch_v1_summary.json"

if csv_path.exists() or json_path.exists():
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    csv_path = triage_dir / f"triage_batch_v1_{stamp}.csv"
    json_path = triage_dir / f"triage_batch_v1_{stamp}.json"
    summary_path = triage_dir / f"triage_batch_v1_{stamp}_summary.json"

triage_results_df.to_csv(csv_path, index=False)
triage_results_df.to_json(json_path, orient="records", indent=2)

completed = triage_results_df[triage_results_df["validation_status"] == "completed"] if not triage_results_df.empty else triage_results_df
invalid = triage_results_df[triage_results_df["validation_status"] == "invalid_response_needs_retry"] if not triage_results_df.empty else triage_results_df
send_to_extraction_ids = []
triage_class_counts = {}
if not completed.empty:
    for _, r in completed.iterrows():
        parsed = r["parsed_triage_response"] or {}
        if parsed.get("send_to_extraction"):
            send_to_extraction_ids.append(r["packet_id"])
        tc = parsed.get("triage_class", "UNKNOWN")
        triage_class_counts[tc] = triage_class_counts.get(tc, 0) + 1

if stage3a_stopped_early:
    stage3a_status = "PROVIDER_UNSTABLE"
elif provider_error_count > 0 or not invalid.empty:
    stage3a_status = "PARTIAL"
elif attempted == 0:
    stage3a_status = "FAIL"
else:
    stage3a_status = "PASS"

summary = {
    "selected_packet_count": len(selected_ids), "attempted_packet_count": attempted,
    "completed_packet_count": len(completed), "provider_error_count": provider_error_count,
    "invalid_response_count": len(invalid), "send_to_extraction_count": len(send_to_extraction_ids),
    "counts_by_municipality": triage_results_df["municipality"].value_counts().to_dict() if not triage_results_df.empty else {},
    "counts_by_variable": triage_results_df["variable_id"].value_counts().to_dict() if not triage_results_df.empty else {},
    "counts_by_triage_class": triage_class_counts,
    "chosen_model_id": CHOSEN_MODEL_ID, "prompt_version": PROMPT_VERSION,
    "run_started_at_utc": run_started_at, "run_finished_at_utc": run_finished_at,
    "stage_status": stage3a_status,
}
summary_path.write_text(json.dumps(summary, indent=2))

print("=" * 72)
print("ZONING PDF PILOT — STAGE 3A: GROQ TRIAGE")
print("=" * 72)
print(f"Selected packets: {len(selected_ids)} / {len(packets_for_triage)}")
print(f"Attempted packets: {attempted}")
print(f"Completed triage results: {len(completed)}")
print(f"Provider errors: {provider_error_count}")
print(f"Invalid responses: {len(invalid)}")
print(f"Packets flagged for possible extraction: {len(send_to_extraction_ids)}")
print()
for muni in PILOT_MUNICIPALITIES:
    print(f"{muni['name']}:")
    for var_id in PILOT_VARIABLES:
        sel_count = sum(1 for pid in selected_ids if pid.startswith(muni["slug"]) and f"_{var_id}_" in pid)
        kept_count = sum(1 for pid in send_to_extraction_ids if pid.startswith(muni["slug"]) and f"_{var_id}_" in pid)
        print(f"  {var_id}: {sel_count} selected; {kept_count} possible extraction")
print()
print(f"Model provider: Groq")
print(f"Model ID: {CHOSEN_MODEL_ID}")
print(f"Prompt version: {PROMPT_VERSION}")
print()
print(f"STAGE 3A RESULT: {stage3a_status}")
print()
print("Meaning:")
print("These are AI triage classifications only. No zoning fact, legal conclusion,")
print("district profile, or score has been produced. Inspect the packet-level")
print("results before approving any later extraction stage.")
print("=" * 72)
print()
print(f"Output CSV: {csv_path.relative_to(PILOT_ROOT)}")
print(f"Output JSON: {json_path.relative_to(PILOT_ROOT)}")
print(f"Output summary: {summary_path.relative_to(PILOT_ROOT)}")
print()
print("STOP: Stage 3B extraction has not started. Human review of triage results is required.")


ZONING PDF PILOT — STAGE 3A: GROQ TRIAGE
Selected packets: 20 / 314
Attempted packets: 8
Completed triage results: 2
Provider errors: 2
Invalid responses: 4
Packets flagged for possible extraction: 2

Greenwich:
  L1: 5 selected; 1 possible extraction
  U2: 5 selected; 1 possible extraction
Bridgeport:
  L1: 5 selected; 0 possible extraction
  U2: 5 selected; 0 possible extraction

Model provider: Groq
Model ID: openai/gpt-oss-20b
Prompt version: triage_v1

STAGE 3A RESULT: PROVIDER_UNSTABLE

Meaning:
These are AI triage classifications only. No zoning fact, legal conclusion,
district profile, or score has been produced. Inspect the packet-level
results before approving any later extraction stage.

Output CSV: outputs/triage/triage_batch_v1_20261002T174857Z.csv
Output JSON: outputs/triage/triage_batch_v1_20261002T174857Z.json
Output summary: outputs/triage/triage_batch_v1_20261002T174857Z_summary.json

STOP: Stage 3B extraction has not started. Human review of triage results is require

---

## Stage 3A v1.1 — Patch and 4-Packet Validation Rerun

The v1 batch run above surfaced three real issues before anything reached a fact sheet:

1. **Two requests were cut off mid-JSON** (`max completion tokens reached before
   generating a valid document`) — the triage call had no output-token budget set.
   Fix: add an explicit `max_completion_tokens` cap.
2. **The model hallucinated a quote** for `greenwich_ct_L1_008` — a plausible-looking
   lot-size value that does not appear anywhere in that packet's actual text (an
   amendment-history table, not the dimensional schedule). The quote-containment
   validator caught this correctly; no code change needed there, but it's worth noting
   this is exactly the failure mode that validator exists for.
3. **The model contradicted its own classification**: `greenwich_ct_L1_021` was labeled
   `DEFINITION_ONLY` but still marked `send_to_extraction = true`. The model should not
   have final say over that field — a deterministic Python rule should override it.

This patch does **not** begin Stage 3B. It fixes the triage mechanism and validates the
fix on exactly four packets — chosen as stress tests, not a full batch:

- `greenwich_ct_L1_007` — a clear table/schedule candidate (Sec. 6-205, contains real
  per-district numeric values) as a positive control.
- `greenwich_ct_L1_010` — wrong-variable stress test (quoted "1 Bedroom Unit 1,350
  square feet," which reads as unit floor area, not lot area).
- `greenwich_ct_L1_021` — definition-only stress test (the triage/routing contradiction
  above).
- `greenwich_ct_U2_001` — the most promising U2 candidate from the v1 run, re-run under
  the patched prompt for comparison.

A full, fresh 20-packet batch only runs after these four are manually inspected.


In [17]:
# --- Stage 3A v1.1 patch: strengthened L1 exclusions, deterministic routing override,
# categorized error types (only genuine provider errors count toward the circuit breaker) ---

TRIAGE_INSTRUCTIONS_V1_1 = TRIAGE_INSTRUCTIONS + """

Do not classify floor area, dwelling-unit size, bedroom-area standards,
minimum habitable area, minimum building area, or minimum unit size
as minimum lot/site area.

Minimum lot/site area requires explicit lot, zoning lot, site, parcel,
tract, or area-per-unit language tied to a land-area requirement."""

# Python-side keyword flag: these terms do not by themselves make a packet irrelevant,
# but they block direct L1 advancement unless lot/site-area language is also present.
FLOOR_AREA_FLAG_TERMS = [
    "bedroom", "unit size", "dwelling unit", "floor area", "gross floor area",
    "habitable area", "minimum building area",
]
LOT_AREA_CONTEXT_TERMS = ["lot area", "zoning lot", "site area", "parcel area", "tract area", "lot size"]


def floor_area_confound_present(variable_id: str, packet_text: str) -> bool:
    if variable_id != "L1":
        return False
    text = packet_text.lower()
    has_floor_area_term = any(t in text for t in FLOOR_AREA_FLAG_TERMS)
    has_lot_area_term = any(t in text for t in LOT_AREA_CONTEXT_TERMS)
    return has_floor_area_term and not has_lot_area_term


# Deterministic overrides applied AFTER the model responds. The model never has final
# say over send_to_extraction — these rules can only turn it False, never True.
AUTO_EXCLUDE_TRIAGE_CLASSES = {
    "DEFINITION_ONLY", "FEE_OR_FINANCIAL_RULE", "CROSS_REFERENCE_ONLY", "IRRELEVANT",
    "REMOVED_RESERVED_OR_EXTERNAL_SOURCE_NOTICE", "UNRESOLVED",
}


def apply_routing_override(parsed: dict, variable_id: str, packet_text: str, quote_valid: bool) -> dict:
    """Returns {"final_send_to_extraction": bool, "routing_override_reason": str|None}."""
    model_value = parsed.get("send_to_extraction")
    triage_class = parsed.get("triage_class")

    if not quote_valid:
        return {"final_send_to_extraction": False,
                "routing_override_reason": "source_quote_failed_containment_validation"}

    if triage_class in AUTO_EXCLUDE_TRIAGE_CLASSES:
        if model_value:
            return {"final_send_to_extraction": False,
                    "routing_override_reason": f"triage_class={triage_class} is auto-excluded regardless of model vote"}
        return {"final_send_to_extraction": False, "routing_override_reason": None}

    if triage_class == "PROCEDURAL_OR_APPROVAL_RULE" and variable_id == "L1":
        return {"final_send_to_extraction": False,
                "routing_override_reason": "PROCEDURAL_OR_APPROVAL_RULE is auto-excluded for L1"}

    if floor_area_confound_present(variable_id, packet_text):
        return {"final_send_to_extraction": False,
                "routing_override_reason": "floor-area/unit-size language present without lot/site-area context"}

    return {"final_send_to_extraction": bool(model_value), "routing_override_reason": None}


# Three distinct error categories. Only "provider_error" counts toward the 25% circuit
# breaker — a cut-off/invalid JSON response is a structured-output failure, not evidence
# that Groq itself is unavailable.
PROVIDER_ERROR_TYPES = ("RateLimitError", "APIConnectionError", "APITimeoutError", "InternalServerError")


def categorize_exception(exc: Exception) -> str:
    name = type(exc).__name__
    status = getattr(exc, "status_code", None)
    if name in PROVIDER_ERROR_TYPES or status in (429, 500, 502, 503, 504):
        return "provider_error"
    return "invalid_structured_output"


print("Stage 3A v1.1 patch defined: strengthened L1 instructions, routing overrides,")
print("floor-area confound flag, and split error categorization.")


Stage 3A v1.1 patch defined: strengthened L1 instructions, routing overrides,
floor-area confound flag, and split error categorization.


In [18]:
# --- Run the patched triage on exactly 4 validation packets (not a full 20-batch) ---
VALIDATION_PACKET_IDS = [
    "greenwich_ct_L1_007",  # positive control: clear table/schedule candidate
    "greenwich_ct_L1_010",  # wrong-variable stress test (unit floor area, not lot area)
    "greenwich_ct_L1_021",  # definition-only stress test
    "greenwich_ct_U2_001",  # most promising U2 candidate, re-run for comparison
]
MAX_COMPLETION_TOKENS = 1024

validation_batch = packets_for_triage[packets_for_triage["packet_id"].isin(VALIDATION_PACKET_IDS)]
print(f"Validation batch: {len(validation_batch)} packets (expected 4).")

validation_run_started_at = datetime.now(timezone.utc).isoformat()
validation_results = []
v_attempted = 0
v_provider_error_count = 0
v_stopped_early = False

if CHOSEN_MODEL_ID is None:
    print("No model selected — skipping validation run.")
else:
    for _, packet_row in validation_batch.iterrows():
        v_attempted += 1
        variable_name = PILOT_VARIABLES[packet_row["variable_target"]]["name"]
        # Patched prompt: use the strengthened L1 instructions in place of the original.
        prompt = build_triage_prompt(packet_row, variable_name).replace(
            TRIAGE_INSTRUCTIONS, TRIAGE_INSTRUCTIONS_V1_1
        )
        request_ts = datetime.now(timezone.utc).isoformat()

        record = {
            "packet_id": packet_row["packet_id"], "municipality": packet_row["municipality"],
            "state": "CT", "variable_id": packet_row["variable_target"],
            "source_document_filename": packet_row["source_filename"],
            "page_range": packet_row["matched_pages"],
            "matched_search_term": packet_row["search_term_matched"],
            "model_provider": "groq", "model_id": CHOSEN_MODEL_ID,
            "prompt_version": "triage_v1.1", "request_timestamp_utc": request_ts,
            "raw_model_response": None, "parsed_triage_response": None,
            "validation_status": None, "validation_issues": None,
            "model_send_to_extraction": None, "final_send_to_extraction": None,
            "routing_override_reason": None, "error_category": None,
        }

        try:
            response = groq_client.chat.completions.create(
                model=CHOSEN_MODEL_ID,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                max_completion_tokens=MAX_COMPLETION_TOKENS,
            )
            raw_text = response.choices[0].message.content
            record["raw_model_response"] = raw_text

            validation = validate_triage_response(raw_text, packet_row["packet_id"], packet_row["extracted_text"])
            record["parsed_triage_response"] = validation["parsed"]
            record["validation_status"] = validation["status"]
            record["validation_issues"] = validation["issues"]

            if validation["status"] == "invalid_response_needs_retry":
                record["error_category"] = "invalid_structured_output"

            if validation["parsed"]:
                quote_ok = "source_quote_not_found_in_packet_text" not in validation["issues"]
                override = apply_routing_override(
                    validation["parsed"], packet_row["variable_target"],
                    packet_row["extracted_text"], quote_ok,
                )
                record["model_send_to_extraction"] = validation["parsed"].get("send_to_extraction")
                record["final_send_to_extraction"] = override["final_send_to_extraction"]
                record["routing_override_reason"] = override["routing_override_reason"]

        except Exception as exc:
            category = categorize_exception(exc)
            record["error_category"] = category
            record["validation_status"] = category
            record["validation_issues"] = [f"{type(exc).__name__}: {str(exc)[:400]}"]
            if category == "provider_error":
                v_provider_error_count += 1

        validation_results.append(record)

        if v_attempted >= 4 and (v_provider_error_count / v_attempted) >= 0.25:
            v_stopped_early = True
            print(f"STAGE_3A_STATUS: PROVIDER_UNSTABLE — stopping after packet {v_attempted}/{len(validation_batch)}.")
            break

validation_run_finished_at = datetime.now(timezone.utc).isoformat()
validation_results_df = pd.DataFrame(validation_results)
print(f"Attempted: {v_attempted}, provider errors: {v_provider_error_count}")
for r in validation_results:
    print(f"\n{r['packet_id']}: validation_status={r['validation_status']}, "
          f"model_send_to_extraction={r['model_send_to_extraction']}, "
          f"final_send_to_extraction={r['final_send_to_extraction']}, "
          f"override_reason={r['routing_override_reason']}")
    if r["parsed_triage_response"]:
        print(f"  triage_class={r['parsed_triage_response'].get('triage_class')}")
        print(f"  quote={r['parsed_triage_response'].get('source_quote_for_triage')!r}")
    if r["validation_issues"]:
        print(f"  issues={r['validation_issues']}")


Validation batch: 4 packets (expected 4).


Attempted: 4, provider errors: 0

greenwich_ct_L1_007: validation_status=invalid_response_needs_retry, model_send_to_extraction=True, final_send_to_extraction=False, override_reason=source_quote_failed_containment_validation
  triage_class=GOVERNING_NUMERIC_STANDARD
  quote='RA-6 2-1/2 35 ft. 7,500 sq. ft.'
  issues=['source_quote_not_found_in_packet_text']

greenwich_ct_L1_010: validation_status=invalid_structured_output, model_send_to_extraction=None, final_send_to_extraction=None, override_reason=None
  issues=['BadRequestError: Error code: 400 - {\'error\': {\'message\': "Failed to generate JSON. Please adjust your prompt. See \'failed_generation\' for more details.", \'type\': \'invalid_request_error\', \'code\': \'json_validate_failed\', \'failed_generation\': \'max completion tokens reached before generating a valid document\'}}']

greenwich_ct_L1_021: validation_status=invalid_structured_output, model_send_to_extraction=None, final_send_to_extraction=None, override_reason=None


In [19]:
# --- Save the v1.1 validation-rerun results (separate file from the v1 batch outputs) ---
val_csv_path = triage_dir / "triage_validation_v1_1.csv"
val_json_path = triage_dir / "triage_validation_v1_1.json"

if val_csv_path.exists() or val_json_path.exists():
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    val_csv_path = triage_dir / f"triage_validation_v1_1_{stamp}.csv"
    val_json_path = triage_dir / f"triage_validation_v1_1_{stamp}.json"

validation_results_df.to_csv(val_csv_path, index=False)
validation_results_df.to_json(val_json_path, orient="records", indent=2)

print("=" * 72)
print("ZONING PDF PILOT — STAGE 3A v1.1: 4-PACKET VALIDATION RERUN")
print("=" * 72)
print(f"Attempted: {v_attempted} / 4")
print(f"Provider errors (true provider/service failures only): {v_provider_error_count}")
print(f"Model ID: {CHOSEN_MODEL_ID}")
print(f"Prompt version: triage_v1.1")
print()
print(f"Output CSV: {val_csv_path.relative_to(PILOT_ROOT)}")
print(f"Output JSON: {val_json_path.relative_to(PILOT_ROOT)}")
print()
print("This is a 4-packet validation rerun only, not a fresh 20-packet batch and not")
print("Stage 3B extraction. Manual inspection of these four records is required before")
print("any further triage batch is run.")
print()
print("STOP: Stage 3B extraction has not started. Human review of these results is required.")


ZONING PDF PILOT — STAGE 3A v1.1: 4-PACKET VALIDATION RERUN
Attempted: 4 / 4
Provider errors (true provider/service failures only): 0
Model ID: openai/gpt-oss-20b
Prompt version: triage_v1.1

Output CSV: outputs/triage/triage_validation_v1_1_20261002T175019Z.csv
Output JSON: outputs/triage/triage_validation_v1_1_20261002T175019Z.json

This is a 4-packet validation rerun only, not a fresh 20-packet batch and not
Stage 3B extraction. Manual inspection of these four records is required before
any further triage batch is run.

STOP: Stage 3B extraction has not started. Human review of these results is required.


---

## Stage 3A Recovery — Diagnostic-First Approach

Part A tests, on exactly one packet, whether removing Groq's strict
`response_format={"type": "json_object"}` constraint (replacing it with explicit
plain-text formatting instructions) resolves the empty/invalid completions seen in v1
and v1.1. This is a testable hypothesis, not an assumed conclusion — the diagnostic
below reports only observed API fields (finish_reason, usage, reasoning-token counts).

Part B — a small 8-packet demonstration batch — only runs if the diagnostic passes.
Neither part begins Stage 3B extraction, and neither overwrites any existing output.


In [20]:
# --- Stage 3A Recovery — Part A: one-packet diagnostic ---
# Hypothesis under test: Groq's strict `response_format={"type":"json_object"}` mode is
# the proximate cause of the earlier empty/invalid completions, not insufficient token
# budget. This is tested, not assumed — the diagnostic below either confirms or refutes
# it for exactly one packet before any batch is attempted.

PROMPT_VERSION_RECOVERY = "triage_recovery_v1"

RECOVERY_L1_GUARDRAIL = """
For L1, do not treat floor area, dwelling-unit size, bedroom-area
standards, habitable area, minimum building area, gross floor area,
or minimum unit size as minimum lot/site area.

A possible L1 rule must explicitly concern lot area, zoning lot area,
site area, parcel area, tract area, or area-per-unit tied to land area.
If that distinction is unclear from the supplied packet, classify it as
UNRESOLVED and do not advance it."""

RECOVERY_U2_GUARDRAIL = """
For U2, do not treat an ADU as two-family permission.
Do not treat a definition alone as proof that two-family housing is permitted.
For form-based codes, a building type is only potentially relevant if the
packet itself supports a two-unit interpretation, such as two principal
dwelling units. If unclear, classify as UNRESOLVED and do not advance it."""

NO_MARKDOWN_INSTRUCTION = """
Return exactly one valid JSON object.
Do not include markdown.
Do not include a code fence.
Do not include explanation before or after the JSON.
Do not include reasoning in the JSON content."""


def build_recovery_prompt(packet_row, variable_name: str) -> str:
    """Same schema/categories as Stage 3A v1.1's build_triage_prompt, but WITHOUT the
    response_format=json_object constraint — formatting is enforced by explicit
    instruction text instead, per the diagnostic hypothesis."""
    guardrail = RECOVERY_L1_GUARDRAIL if packet_row["variable_target"] == "L1" else RECOVERY_U2_GUARDRAIL
    base_prompt = build_triage_prompt(packet_row, variable_name).replace(
        TRIAGE_INSTRUCTIONS, TRIAGE_INSTRUCTIONS + guardrail
    )
    return base_prompt + "\n" + NO_MARKDOWN_INSTRUCTION


def run_diagnostic_request(packet_row, variable_name: str) -> dict:
    """One request, no response_format enforcement. Records only observed fields —
    no speculation about why a given outcome occurred."""
    prompt = build_recovery_prompt(packet_row, variable_name)
    request_ts = datetime.now(timezone.utc).isoformat()
    diag = {
        "packet_id": packet_row["packet_id"], "model_id": CHOSEN_MODEL_ID,
        "prompt_version": PROMPT_VERSION_RECOVERY,
        "request_configuration_without_secrets": {
            "model": CHOSEN_MODEL_ID, "temperature": 0.0, "max_completion_tokens": 2048,
            "response_format": "none (plain text + explicit formatting instruction)",
        },
        "request_timestamp_utc": request_ts, "api_completed": False,
        "finish_reason_if_available": None, "usage_fields_if_available": None,
        "reasoning_fields_if_available": None, "content_length": 0,
        "diagnostic_outcome": None, "raw_response_preview_max_1500_characters": None,
        "json_parse_result": None, "schema_validation_result": None,
        "quote_containment_result": None, "routing_override_result": None,
    }

    try:
        response = groq_client.chat.completions.create(
            model=CHOSEN_MODEL_ID,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_completion_tokens=2048,
        )
        diag["api_completed"] = True
        choice = response.choices[0]
        diag["finish_reason_if_available"] = choice.finish_reason
        usage = response.usage
        if usage:
            diag["usage_fields_if_available"] = {
                "completion_tokens": usage.completion_tokens, "prompt_tokens": usage.prompt_tokens,
                "total_tokens": usage.total_tokens,
            }
            reasoning_tokens = getattr(getattr(usage, "completion_tokens_details", None), "reasoning_tokens", None)
            diag["reasoning_fields_if_available"] = {"reasoning_tokens": reasoning_tokens}
        content = choice.message.content or ""
        diag["content_length"] = len(content)
        diag["raw_response_preview_max_1500_characters"] = content[:1500]

        if not content.strip():
            diag["diagnostic_outcome"] = "EMPTY_COMPLETION"
        else:
            try:
                parsed = json.loads(content)
                diag["json_parse_result"] = "success"
                validation = validate_triage_response(content, packet_row["packet_id"], packet_row["extracted_text"])
                diag["schema_validation_result"] = validation["status"]
                diag["quote_containment_result"] = (
                    "fail" if "source_quote_not_found_in_packet_text" in validation["issues"] else "pass"
                )
                if validation["status"] == "completed":
                    quote_ok = diag["quote_containment_result"] == "pass"
                    override = apply_routing_override(
                        validation["parsed"], packet_row["variable_target"],
                        packet_row["extracted_text"], quote_ok,
                    )
                    diag["routing_override_result"] = override
                    diag["diagnostic_outcome"] = "DIAGNOSTIC_PASS"
                elif diag["quote_containment_result"] == "fail":
                    diag["diagnostic_outcome"] = "QUOTE_CONTAINMENT_FAILURE"
                else:
                    diag["diagnostic_outcome"] = "SCHEMA_VALIDATION_FAILURE"
            except json.JSONDecodeError:
                diag["json_parse_result"] = "failed"
                diag["diagnostic_outcome"] = "NON_JSON_COMPLETION"
    except Exception as exc:
        diag["diagnostic_outcome"] = "API_ERROR"
        diag["raw_response_preview_max_1500_characters"] = f"{type(exc).__name__}: {str(exc)[:400]}"

    return diag


diagnostic_packet = packets_for_triage[packets_for_triage["packet_id"] == "greenwich_ct_L1_007"].iloc[0]
diagnostic_result = run_diagnostic_request(diagnostic_packet, PILOT_VARIABLES["L1"]["name"])

print(f"Diagnostic packet: {diagnostic_result['packet_id']}")
print(f"API completed: {diagnostic_result['api_completed']}")
print(f"Finish reason: {diagnostic_result['finish_reason_if_available']}")
print(f"Usage: {diagnostic_result['usage_fields_if_available']}")
print(f"Reasoning fields: {diagnostic_result['reasoning_fields_if_available']}")
print(f"Content length: {diagnostic_result['content_length']}")
print(f"Diagnostic outcome: {diagnostic_result['diagnostic_outcome']}")
print(f"Routing override result: {diagnostic_result['routing_override_result']}")
print(f"\nRaw response preview:\n{diagnostic_result['raw_response_preview_max_1500_characters']}")


Diagnostic packet: greenwich_ct_L1_007
API completed: True
Finish reason: stop
Usage: {'completion_tokens': 594, 'prompt_tokens': 4189, 'total_tokens': 4783}
Reasoning fields: {'reasoning_tokens': 407}
Content length: 633
Diagnostic outcome: DIAGNOSTIC_PASS
Routing override result: {'final_send_to_extraction': True, 'routing_override_reason': None}

Raw response preview:
{"packet_id":"greenwich_ct_L1_007","triage_class":"GOVERNING_NUMERIC_STANDARD","target_variable_relevance":"direct","likely_governing_rule":true,"send_to_extraction":true,"districts_or_building_types_mentioned":["RA-4","RA-2","RA-1","R-20","R-12","R-7","R-6","R-MF"],"numeric_standard_present":true,"permission_pathway_present":false,"table_or_schedule_present":true,"footnote_or_exception_present":true,"cross_reference_present":true,"short_reason":"Table lists minimum lot sizes for residential zones.","follow_up_needed":false,"follow_up_reason":null,"source_quote_for_triage":"RA-4 3-1/2 50 ft. 4 Acres","triage_confidence

In [21]:
# --- Save the diagnostic result; print the "diagnostic failed" block if applicable ---
recovery_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
diag_path = triage_dir / f"triage_recovery_diagnostic_{recovery_stamp}.json"
diag_path.write_text(json.dumps(diagnostic_result, indent=2, default=str))
print(f"Diagnostic output saved: {diag_path.relative_to(PILOT_ROOT)}")

DIAGNOSTIC_PASSED = diagnostic_result["diagnostic_outcome"] == "DIAGNOSTIC_PASS"

if not DIAGNOSTIC_PASSED:
    print()
    print("=" * 72)
    print("ZONING PDF PILOT — STAGE 3A RECOVERY DIAGNOSTIC")
    print("=" * 72)
    print(f"Packet tested: {diagnostic_result['packet_id']}")
    print(f"Model: {CHOSEN_MODEL_ID}")
    print("Configuration tested: plain JSON (no response_format)")
    print(f"Diagnostic outcome: {diagnostic_result['diagnostic_outcome']}")
    print("Batch triage run: NOT STARTED")
    print()
    print("Meaning:")
    print("The Stage 3A model/output configuration did not produce a reliable,")
    print("validated result for one controlled source packet. No extraction,")
    print("fact-sheet record, score, or legal conclusion was produced.")
    print("=" * 72)


Diagnostic output saved: outputs/triage/triage_recovery_diagnostic_20261002T175052Z.json


In [22]:
# --- Stage 3A Recovery — Part B: 8-packet demonstration batch (only if diagnostic passed) ---
RECOVERY_PACKET_IDS = []
recovery_batch_results = []
recovery_run_started_at = None
recovery_run_finished_at = None

if not DIAGNOSTIC_PASSED:
    print("Diagnostic did not pass — batch triage not started.")
else:
    REQUIRED_RECOVERY_IDS = [
        "greenwich_ct_L1_010", "greenwich_ct_L1_021", "greenwich_ct_U2_001", "bridgeport_ct_U2_003",
    ]
    # Fill remaining slots to reach 2 per (municipality, variable) bucket, minimizing
    # duplicate page ranges, using the same scoring function as the original selection.
    used_pages = set()
    for pid in REQUIRED_RECOVERY_IDS:
        row = packets_for_triage[packets_for_triage["packet_id"] == pid]
        if not row.empty:
            used_pages.add(row.iloc[0]["matched_pages"])

    buckets = [("Greenwich", "L1"), ("Greenwich", "U2"), ("Bridgeport", "L1"), ("Bridgeport", "U2")]
    bucket_counts = {b: sum(1 for pid in REQUIRED_RECOVERY_IDS
                             if packets_for_triage.loc[packets_for_triage["packet_id"] == pid, "municipality"].eq(b[0]).any()
                             and packets_for_triage.loc[packets_for_triage["packet_id"] == pid, "variable_target"].eq(b[1]).any())
                      for b in buckets}

    fill_ids = []
    for muni_name, var_id in buckets:
        needed = 2 - bucket_counts[(muni_name, var_id)]
        if needed <= 0:
            continue
        bucket_df = packets_for_triage[
            (packets_for_triage["municipality"] == muni_name)
            & (packets_for_triage["variable_target"] == var_id)
            & (~packets_for_triage["packet_id"].isin(REQUIRED_RECOVERY_IDS + fill_ids))
        ].copy()
        bucket_df["_score"] = bucket_df.apply(score_packet, axis=1)
        bucket_df = bucket_df.sort_values("_score")
        for _, row in bucket_df.iterrows():
            if needed <= 0:
                break
            if row["matched_pages"] in used_pages:
                continue  # minimize duplicate page ranges
            fill_ids.append(row["packet_id"])
            used_pages.add(row["matched_pages"])
            needed -= 1

    RECOVERY_PACKET_IDS = REQUIRED_RECOVERY_IDS + fill_ids
    print(f"Selected {len(RECOVERY_PACKET_IDS)} packets for the recovery batch:")
    for pid in RECOVERY_PACKET_IDS:
        tag = "required" if pid in REQUIRED_RECOVERY_IDS else "fill (lowest-redundancy)"
        print(f"  {pid}: {tag}")

    recovery_batch = packets_for_triage[packets_for_triage["packet_id"].isin(RECOVERY_PACKET_IDS)]
    recovery_run_started_at = datetime.now(timezone.utc).isoformat()

    for _, packet_row in recovery_batch.iterrows():
        variable_name = PILOT_VARIABLES[packet_row["variable_target"]]["name"]
        prompt = build_recovery_prompt(packet_row, variable_name)
        request_ts = datetime.now(timezone.utc).isoformat()

        record = {
            "packet_id": packet_row["packet_id"], "municipality": packet_row["municipality"],
            "variable_id": packet_row["variable_target"], "page_range": packet_row["matched_pages"],
            "matched_search_term": packet_row["search_term_matched"],
            "triage_class": None, "target_variable_relevance": None,
            "model_send_to_extraction": None, "final_send_to_extraction": None,
            "routing_override_reason": None, "source_quote_for_triage": None,
            "quote_containment_status": None, "triage_confidence": None,
            "triage_status": None, "validation_issues": None,
            "model_id": CHOSEN_MODEL_ID, "prompt_version": PROMPT_VERSION_RECOVERY,
            "request_timestamp_utc": request_ts, "raw_model_response": None,
        }

        try:
            response = groq_client.chat.completions.create(
                model=CHOSEN_MODEL_ID,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_completion_tokens=2048,
            )
            content = response.choices[0].message.content or ""
            record["raw_model_response"] = content

            if not content.strip():
                record["triage_status"] = "empty_completion"
            else:
                try:
                    json.loads(content)
                    validation = validate_triage_response(content, packet_row["packet_id"], packet_row["extracted_text"])
                    quote_ok = "source_quote_not_found_in_packet_text" not in validation["issues"]
                    record["quote_containment_status"] = "pass" if quote_ok else "fail"
                    record["triage_status"] = validation["status"]
                    record["validation_issues"] = validation["issues"]
                    if validation["parsed"]:
                        p = validation["parsed"]
                        record["triage_class"] = p.get("triage_class")
                        record["target_variable_relevance"] = p.get("target_variable_relevance")
                        record["model_send_to_extraction"] = p.get("send_to_extraction")
                        record["source_quote_for_triage"] = p.get("source_quote_for_triage")
                        record["triage_confidence"] = p.get("triage_confidence")
                        override = apply_routing_override(p, packet_row["variable_target"], packet_row["extracted_text"], quote_ok)
                        record["final_send_to_extraction"] = override["final_send_to_extraction"]
                        record["routing_override_reason"] = override["routing_override_reason"]
                except json.JSONDecodeError:
                    record["triage_status"] = "non_json_completion"
        except Exception as exc:
            record["triage_status"] = "api_error"
            record["validation_issues"] = [f"{type(exc).__name__}: {str(exc)[:400]}"]

        recovery_batch_results.append(record)

    recovery_run_finished_at = datetime.now(timezone.utc).isoformat()
    print(f"\nAttempted {len(recovery_batch_results)} packets.")


Selected 8 packets for the recovery batch:
  greenwich_ct_L1_010: required
  greenwich_ct_L1_021: required
  greenwich_ct_U2_001: required
  bridgeport_ct_U2_003: required
  greenwich_ct_U2_019: fill (lowest-redundancy)
  bridgeport_ct_L1_032: fill (lowest-redundancy)
  bridgeport_ct_L1_033: fill (lowest-redundancy)
  bridgeport_ct_U2_046: fill (lowest-redundancy)



Attempted 8 packets.


In [23]:
# --- Save recovery-batch outputs and print the required summary ---
if DIAGNOSTIC_PASSED and recovery_batch_results:
    recovery_df = pd.DataFrame(recovery_batch_results)
    batch_csv_path = triage_dir / f"triage_recovery_batch_{recovery_stamp}.csv"
    batch_json_path = triage_dir / f"triage_recovery_batch_{recovery_stamp}.json"
    summary_json_path = triage_dir / f"triage_recovery_summary_{recovery_stamp}.json"

    recovery_df.to_csv(batch_csv_path, index=False)
    recovery_df.to_json(batch_json_path, orient="records", indent=2)

    status_counts = recovery_df["triage_status"].value_counts().to_dict()
    completed = recovery_df[recovery_df["triage_status"] == "completed"]
    kept = recovery_df[recovery_df["final_send_to_extraction"] == True]

    by_bucket = {}
    for muni_name, var_id in [("Greenwich", "L1"), ("Greenwich", "U2"), ("Bridgeport", "L1"), ("Bridgeport", "U2")]:
        by_bucket[f"{muni_name} {var_id}"] = len(recovery_df[
            (recovery_df["municipality"] == muni_name) & (recovery_df["variable_id"] == var_id)
        ])

    recovery_result = "PASS" if len(completed) == len(recovery_df) else "PARTIAL"

    summary = {
        "diagnostic_packet": diagnostic_result["packet_id"], "diagnostic_outcome": diagnostic_result["diagnostic_outcome"],
        "selected_packet_count": len(RECOVERY_PACKET_IDS), "attempted_packet_count": len(recovery_df),
        "completed_packet_count": len(completed),
        "api_error_count": status_counts.get("api_error", 0),
        "empty_completion_count": status_counts.get("empty_completion", 0),
        "non_json_completion_count": status_counts.get("non_json_completion", 0),
        "schema_validation_failure_count": status_counts.get("invalid_response_needs_retry", 0),
        "quote_containment_failure_count": int((recovery_df["quote_containment_status"] == "fail").sum()),
        "final_send_to_extraction_true_count": len(kept),
        "counts_by_municipality_variable": by_bucket,
        "model_id": CHOSEN_MODEL_ID, "prompt_version": PROMPT_VERSION_RECOVERY,
        "run_started_at_utc": recovery_run_started_at, "run_finished_at_utc": recovery_run_finished_at,
        "stage_status": recovery_result,
    }
    summary_json_path.write_text(json.dumps(summary, indent=2))

    print("=" * 72)
    print("ZONING PDF PILOT — STAGE 3A RECOVERY BATCH")
    print("=" * 72)
    print(f"Diagnostic packet: {diagnostic_result['packet_id']}")
    print(f"Diagnostic outcome: {diagnostic_result['diagnostic_outcome']}")
    print()
    print(f"Selected packets: {len(RECOVERY_PACKET_IDS)}")
    print(f"Attempted packets: {len(recovery_df)}")
    print(f"Completed triage results: {len(completed)}")
    print(f"API errors: {status_counts.get('api_error', 0)}")
    print(f"Empty completions: {status_counts.get('empty_completion', 0)}")
    print(f"Non-JSON completions: {status_counts.get('non_json_completion', 0)}")
    print(f"Schema validation failures: {status_counts.get('invalid_response_needs_retry', 0)}")
    print(f"Quote containment failures: {int((recovery_df['quote_containment_status'] == 'fail').sum())}")
    print(f"Packets with final_send_to_extraction = true: {len(kept)}")
    print()
    print("By municipality and variable:")
    for k, v in by_bucket.items():
        print(f"{k}: {v}")
    print()
    print(f"Model: {CHOSEN_MODEL_ID}")
    print(f"Prompt version: {PROMPT_VERSION_RECOVERY}")
    print()
    print(f"STAGE 3A RECOVERY RESULT: {recovery_result}")
    print()
    print("Meaning:")
    print("These are source-linked AI triage results only. No final zoning fact,")
    print("legal conclusion, extraction record, human-review decision, or score")
    print("has been produced.")
    print("=" * 72)
    print()
    print(f"Diagnostic JSON: {diag_path.relative_to(PILOT_ROOT)}")
    print(f"Batch CSV: {batch_csv_path.relative_to(PILOT_ROOT)}")
    print(f"Batch JSON: {batch_json_path.relative_to(PILOT_ROOT)}")
    print(f"Summary JSON: {summary_json_path.relative_to(PILOT_ROOT)}")
    print()
    print("STOP: Stage 3B extraction has not started. Review the recovery-batch triage outputs before proceeding.")
else:
    print("STOP: Stage 3B extraction has not started. Review the recovery-batch triage outputs before proceeding.")


ZONING PDF PILOT — STAGE 3A RECOVERY BATCH
Diagnostic packet: greenwich_ct_L1_007
Diagnostic outcome: DIAGNOSTIC_PASS

Selected packets: 8
Attempted packets: 8
Completed triage results: 6
API errors: 0
Empty completions: 0
Non-JSON completions: 0
Schema validation failures: 2
Quote containment failures: 2
Packets with final_send_to_extraction = true: 3

By municipality and variable:
Greenwich L1: 2
Greenwich U2: 2
Bridgeport L1: 2
Bridgeport U2: 2

Model: openai/gpt-oss-20b
Prompt version: triage_recovery_v1

STAGE 3A RECOVERY RESULT: PARTIAL

Meaning:
These are source-linked AI triage results only. No final zoning fact,
legal conclusion, extraction record, human-review decision, or score
has been produced.

Diagnostic JSON: outputs/triage/triage_recovery_diagnostic_20261002T175052Z.json
Batch CSV: outputs/triage/triage_recovery_batch_20261002T175052Z.csv
Batch JSON: outputs/triage/triage_recovery_batch_20261002T175052Z.json
Summary JSON: outputs/triage/triage_recovery_summary_20261002

---

## Stage 3A v1.2 — Final Local Safeguard Validation

This is a **local, offline correction** of the existing recovery-batch results already
saved to disk. It makes **zero LLM/API calls** — it only re-evaluates the saved model
outputs against a stronger deterministic L1 exclusion rule and the full packet context,
then validates every record before declaring Stage 3A complete or not.

It does not create new model outputs, new zoning facts, candidate evidence records,
review records, scores, or extraction results.


In [1]:
# --- Stage 3A v1.2: local correction, zero API calls ---
# Loads the existing recovery-batch JSON plus Stage 2 packet text and Stage 1 manifest
# metadata already on disk, and re-evaluates routing with a stronger L1 exclusion rule.
# Self-contained: does not depend on any earlier cell's in-memory state, so this section
# can be executed on its own without re-running (and re-calling the API from) Stage 3A's
# earlier Groq-calling cells.
import re
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

PILOT_ROOT = Path.cwd()
if not (PILOT_ROOT / "data").exists() and (PILOT_ROOT.parent / "zoning_pdf_pilot" / "data").exists():
    PILOT_ROOT = PILOT_ROOT.parent / "zoning_pdf_pilot"
triage_dir = PILOT_ROOT / "outputs" / "triage"

v12_run_started_at = datetime.now(timezone.utc).isoformat()

recovery_batch_path = PILOT_ROOT / "outputs/triage/triage_recovery_batch_20261002T175052Z.json"
recovery_batch_v1 = json.load(open(recovery_batch_path))
source_packets_by_id = {p["packet_id"]: p for p in json.load(open(PILOT_ROOT / "data/processed/source_packets/source_packets_v1.json"))}
manifest_by_filename = {m["local_filename"]: m for m in json.load(open(PILOT_ROOT / "data/source_manifest/source_manifest_v1.json"))}

UNIT_TYPE_PHRASES = [
    "room efficiency", "efficiency unit", "efficiency apartment", "efficiency dwelling",
    "studio", "studio unit", "studio apartment", "one-room", "1-room", "single-room",
    "single room", "single-room occupancy", "sro", "dormitory", "boarding house",
    "rooming house", "rooming unit", "sleeping unit", "living unit", "apartment unit",
    "bedroom unit",
]
PER_UNIT_PHRASES = [
    "per room", "per unit", "per dwelling", "per dwelling unit", "per bedroom",
    "per occupant", "per person", "per resident", "per bed", "unit size", "dwelling size",
    "apartment size", "minimum unit area", "minimum dwelling area", "minimum living area",
    "minimum habitable area", "living area", "habitable area", "gross floor area per",
    "net floor area per",
]
OTHER_LEGAL_FORMS = [
    "floor area ratio", "far", "lot coverage", "building coverage", "building footprint",
    "open space", "usable area", "building area", "gross floor area", "net floor area",
    "density",
]
LAND_AREA_CONTEXT_TERMS = ["lot area", "zoning lot", "site area", "parcel area", "tract area", "lot size"]
ALL_L1_FLAG_PHRASES = UNIT_TYPE_PHRASES + PER_UNIT_PHRASES + OTHER_LEGAL_FORMS

L1_REGEX_PATTERNS = [
    re.compile(r"\b\d[\d,\.]*\s*(sq\.?\s*ft\.?|square feet|sq\.?\s*ft|sf)\s*(per|/)\s*(room|unit|dwelling|bedroom|person|occupant|resident|bed)\b", re.I),
    re.compile(r"\b(efficiency|studio|one[- ]room|1[- ]room|single[- ]room)\b.*\b\d[\d,\.]*\s*(sq\.?\s*ft\.?|square feet|sq\.?\s*ft|sf)\b", re.I),
    re.compile(r"\b\d[\d,\.]*\s*(sq\.?\s*ft\.?|square feet|sq\.?\s*ft|sf)\b.*\b(efficiency|studio|per room|per unit|per bedroom|per person|per occupant|per resident|per bed)\b", re.I),
]


def l1_floor_area_signal_present(quote, packet_text) -> bool:
    for h in [(quote or "").lower(), (packet_text or "").lower()]:
        if any(phrase in h for phrase in ALL_L1_FLAG_PHRASES):
            return True
        if any(p.search(h) for p in L1_REGEX_PATTERNS):
            return True
    return False


def l1_land_area_context_present(quote, packet_text) -> bool:
    for h in [(quote or "").lower(), (packet_text or "").lower()]:
        if any(term in h for term in LAND_AREA_CONTEXT_TERMS):
            return True
    return False


AUTO_EXCLUDE_CLASSES_V12 = {
    "DEFINITION_ONLY", "FEE_OR_FINANCIAL_RULE", "CROSS_REFERENCE_ONLY", "IRRELEVANT",
    "REMOVED_RESERVED_OR_EXTERNAL_SOURCE_NOTICE", "UNRESOLVED",
}


def normalize_ws_v12(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip().lower()


corrected_records = []
for rec in recovery_batch_v1:
    pkt = source_packets_by_id.get(rec["packet_id"], {})
    packet_text = pkt.get("extracted_text", "")
    manifest_entry = manifest_by_filename.get(pkt.get("source_filename", ""), {})

    reasons = []
    issues = list(rec.get("validation_issues") or [])

    quote = rec.get("source_quote_for_triage")
    quote_ok = rec.get("quote_containment_status") == "pass"
    if quote and normalize_ws_v12(quote) not in normalize_ws_v12(packet_text):
        quote_ok = False
        if "source_quote_not_found_in_packet_text" not in issues:
            issues.append("source_quote_not_found_in_packet_text")
    if not quote_ok:
        reasons.append("quote_containment_failure")

    triage_class = rec.get("triage_class")
    if triage_class in AUTO_EXCLUDE_CLASSES_V12:
        reasons.append(f"auto_excluded_class_{str(triage_class).lower()}")
    elif triage_class == "PROCEDURAL_OR_APPROVAL_RULE" and rec["variable_id"] == "L1":
        reasons.append("procedural_rule_excluded_for_l1")

    if rec.get("triage_status") != "completed":
        reasons.append(f"triage_status_{rec.get('triage_status')}")

    conf = rec.get("triage_confidence")
    if conf is not None and not (isinstance(conf, (int, float)) and 0.0 <= conf <= 1.0):
        reasons.append("confidence_out_of_range")

    l1_land_context_note = None
    if rec["variable_id"] == "L1":
        has_floor_signal = l1_floor_area_signal_present(quote, packet_text)
        has_land_context = l1_land_area_context_present(quote, packet_text)
        if has_floor_signal and not has_land_context:
            reasons.append("floor_area_or_unit_size_exclusion")
        elif has_floor_signal and has_land_context:
            # Flagged by the floor-area/unit-type wordlist, but the packet's own text
            # also contains explicit land-area language — per the rule, this is NOT
            # auto-excluded. Recorded as a note for human review, not a block.
            l1_land_context_note = (
                "floor_area_or_unit_size_phrase_present_but_packet_also_contains_"
                "explicit_land_area_language_not_auto_excluded"
            )

    final_v12 = (len(reasons) == 0) and bool(rec.get("model_send_to_extraction"))

    corrected = dict(rec)
    corrected["original_final_send_to_extraction"] = rec.get("final_send_to_extraction")
    corrected["original_routing_override_reason"] = rec.get("routing_override_reason")
    corrected["stage3a_v1_2_final_send_to_extraction"] = final_v12
    corrected["stage3a_v1_2_routing_override_reasons"] = reasons if reasons else None
    corrected["stage3a_v1_2_validation_issues"] = issues if issues else None
    corrected["stage3a_v1_2_land_context_note"] = l1_land_context_note
    corrected["stage3a_v1_2_correction_applied"] = True
    corrected["stage3a_v1_2_corrected_at_utc"] = datetime.now(timezone.utc).isoformat()
    corrected["state"] = manifest_entry.get("state", "CT")
    corrected["source_document_filename"] = pkt.get("source_filename")
    corrected["document_sha256"] = manifest_entry.get("raw_pdf_sha256")

    corrected_records.append(corrected)

v12_run_finished_at = datetime.now(timezone.utc).isoformat()
corrected_by_id = {r["packet_id"]: r for r in corrected_records}
print(f"Loaded and corrected {len(corrected_records)} records. Zero API calls made.")


Loaded and corrected 8 records. Zero API calls made.


In [2]:
# --- Required scenario checks and completion gate ---
# These check mechanically against the actual corrected data. A check that fails here
# is reported as a failure, with the underlying reason shown — not adjusted to force a
# particular outcome.

def scenario_check(pid, expect_false=None, expect_true=None, must_include_reason=None):
    r = corrected_by_id.get(pid)
    if r is None:
        return False, "packet not found"
    val = r["stage3a_v1_2_final_send_to_extraction"]
    if expect_false is not None and val is not False:
        return False, f"expected False, got {val}"
    if expect_true is not None and val is not True:
        return False, f"expected True, got {val}"
    if must_include_reason:
        reasons = r["stage3a_v1_2_routing_override_reasons"] or []
        if must_include_reason not in reasons:
            return False, f"expected reason '{must_include_reason}' not present (got {reasons})"
    return True, None


scenario_results = {}
scenario_results["greenwich_ct_L1_010"] = scenario_check("greenwich_ct_L1_010", expect_false=True, must_include_reason="floor_area_or_unit_size_exclusion")
scenario_results["bridgeport_ct_L1_032"] = scenario_check("bridgeport_ct_L1_032", expect_false=True, must_include_reason="quote_containment_failure")
scenario_results["bridgeport_ct_L1_033"] = scenario_check("bridgeport_ct_L1_033", expect_false=True, must_include_reason="quote_containment_failure")
scenario_results["greenwich_ct_L1_021"] = scenario_check("greenwich_ct_L1_021", expect_false=True)
scenario_results["greenwich_ct_U2_001"] = scenario_check("greenwich_ct_U2_001", expect_true=True)
scenario_results["bridgeport_ct_U2_003"] = scenario_check("bridgeport_ct_U2_003", expect_true=True)

for pid, (passed, detail) in scenario_results.items():
    status = "PASS" if passed else f"FAIL ({detail})"
    print(f"{pid}: {status}")

# If greenwich_ct_L1_010 failed its expected-false check, inspect why before accepting
# the gate result — this packet's source text turned out to matter.
if not scenario_results["greenwich_ct_L1_010"][0]:
    print()
    print("NOTE on greenwich_ct_L1_010: re-reading the full packet text shows a second")
    print("table on the same page, under Sec. 6-71.1 (an affordable-housing modification")
    print("to the R-PHD-SU zone), explicitly titled 'Minimum Lot Area Required For Each")
    print("Type of Unit' — with '1 Room Efficiency ... 1,350 square feet' as a row in")
    print("THAT table, not the earlier floor-area table. The quote is literally present")
    print("and is explicitly land-area language, so the deterministic rule (quote OR")
    print("packet context establishes land-area tie) correctly does NOT exclude it.")
    print("Whether a zone-specific affordable-housing modification counts as Greenwich's")
    print("general L1 rule is a legal-scope question for human review, not something")
    print("either the model or this keyword rule should decide unilaterally.")

completion_gate_passed = all(passed for passed, _ in scenario_results.values())


greenwich_ct_L1_010: FAIL (expected False, got True)
bridgeport_ct_L1_032: PASS
bridgeport_ct_L1_033: PASS
greenwich_ct_L1_021: PASS
greenwich_ct_U2_001: PASS
bridgeport_ct_U2_003: PASS

NOTE on greenwich_ct_L1_010: re-reading the full packet text shows a second
table on the same page, under Sec. 6-71.1 (an affordable-housing modification
to the R-PHD-SU zone), explicitly titled 'Minimum Lot Area Required For Each
Type of Unit' — with '1 Room Efficiency ... 1,350 square feet' as a row in
THAT table, not the earlier floor-area table. The quote is literally present
and is explicitly land-area language, so the deterministic rule (quote OR
packet context establishes land-area tie) correctly does NOT exclude it.
Whether a zone-specific affordable-housing modification counts as Greenwich's
general L1 rule is a legal-scope question for human review, not something
either the model or this keyword rule should decide unilaterally.


In [3]:
# --- Save v1.2 outputs (new timestamped files only) and print the final summary ---
v12_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
v12_csv_path = triage_dir / f"triage_stage3a_final_v1_2_{v12_stamp}.csv"
v12_json_path = triage_dir / f"triage_stage3a_final_v1_2_{v12_stamp}.json"
v12_summary_path = triage_dir / f"triage_stage3a_final_v1_2_summary_{v12_stamp}.json"

corrected_df = pd.DataFrame(corrected_records)
display_cols = ["packet_id", "municipality", "variable_id", "page_range", "matched_search_term",
                 "triage_class", "source_quote_for_triage", "quote_containment_status",
                 "model_send_to_extraction", "original_final_send_to_extraction",
                 "stage3a_v1_2_final_send_to_extraction", "stage3a_v1_2_routing_override_reasons",
                 "stage3a_v1_2_validation_issues", "triage_confidence", "model_id",
                 "prompt_version", "request_timestamp_utc"]
corrected_df[display_cols].to_csv(v12_csv_path, index=False)
corrected_df.to_json(v12_json_path, orient="records", indent=2)

final_true = [r["packet_id"] for r in corrected_records if r["stage3a_v1_2_final_send_to_extraction"]]
final_false = [r for r in corrected_records if not r["stage3a_v1_2_final_send_to_extraction"]]
blocked_quote = sum(1 for r in corrected_records if r["stage3a_v1_2_routing_override_reasons"] and "quote_containment_failure" in r["stage3a_v1_2_routing_override_reasons"])
blocked_floor = sum(1 for r in corrected_records if r["stage3a_v1_2_routing_override_reasons"] and "floor_area_or_unit_size_exclusion" in r["stage3a_v1_2_routing_override_reasons"])
blocked_def = sum(1 for r in corrected_records if r["stage3a_v1_2_routing_override_reasons"] and any(x.startswith("auto_excluded_class") for x in r["stage3a_v1_2_routing_override_reasons"]))
blocked_other = sum(1 for r in corrected_records if r["stage3a_v1_2_routing_override_reasons"]) - blocked_quote - blocked_floor - blocked_def

stage3a_final_result = "PASS — READY FOR STAGE 3B PLANNING" if completion_gate_passed else "PARTIAL — NOT READY FOR STAGE 3B"

summary = {
    "records_loaded": len(recovery_batch_v1), "records_corrected": len(corrected_records),
    "records_with_final_advancement_true": len(final_true), "records_with_final_advancement_false": len(final_false),
    "records_blocked_by_quote_containment": blocked_quote, "records_blocked_by_floor_area_or_unit_size": blocked_floor,
    "records_blocked_by_definition_only": blocked_def, "records_blocked_by_other_safeguards": max(blocked_other, 0),
    "final_possible_extraction_packet_ids": final_true, "completion_gate_passed": completion_gate_passed,
    "stage3a_final_result": stage3a_final_result, "llm_api_calls_made": 0,
    "run_started_at_utc": v12_run_started_at, "run_finished_at_utc": v12_run_finished_at,
    "scenario_check_results": {k: v[0] for k, v in scenario_results.items()},
}
v12_summary_path.write_text(json.dumps(summary, indent=2))

print("=" * 72)
print("ZONING PDF PILOT — STAGE 3A v1.2 FINAL SAFEGUARD VALIDATION")
print("=" * 72)
print("LLM/API calls made: 0")
print(f"Recovery records loaded: {len(recovery_batch_v1)}")
print(f"Records corrected by v1.2 safeguards: {len(corrected_records)}")
print()
print("Blocked by:")
print(f"  Quote-containment failure: {blocked_quote}")
print(f"  Floor-area/unit-size exclusion: {blocked_floor}")
print(f"  Definition-only rule: {blocked_def}")
print(f"  Other safeguards: {max(blocked_other, 0)}")
print()
print(f"Final possible-extraction packets: {final_true if final_true else 'none'}")
print()
print("Required scenario checks:")
for pid, (passed, detail) in scenario_results.items():
    print(f"  {pid}: {'PASS' if passed else 'FAIL'}")
print()
print(f"STAGE 3A FINAL RESULT: {stage3a_final_result}")
print()
print("Meaning:")
print("All results remain AI triage only. No verified zoning fact, legal conclusion,")
print("candidate evidence record, human-review decision, subindex, composite, or")
print("score has been produced.")
print("=" * 72)
print()
print(f"CSV: {v12_csv_path.relative_to(PILOT_ROOT)}")
print(f"JSON: {v12_json_path.relative_to(PILOT_ROOT)}")
print(f"Summary: {v12_summary_path.relative_to(PILOT_ROOT)}")
print()
if completion_gate_passed:
    print("STOP: Stage 3A is complete and ready for Stage 3B planning. Stage 3B extraction has not started.")
else:
    print("STOP: Stage 3A remains partial. Stage 3B extraction has not started.")


ZONING PDF PILOT — STAGE 3A v1.2 FINAL SAFEGUARD VALIDATION
LLM/API calls made: 0
Recovery records loaded: 8
Records corrected by v1.2 safeguards: 8

Blocked by:
  Quote-containment failure: 2
  Floor-area/unit-size exclusion: 0
  Definition-only rule: 2
  Other safeguards: 0

Final possible-extraction packets: ['greenwich_ct_L1_010', 'greenwich_ct_U2_001', 'bridgeport_ct_U2_003']

Required scenario checks:
  greenwich_ct_L1_010: FAIL
  bridgeport_ct_L1_032: PASS
  bridgeport_ct_L1_033: PASS
  greenwich_ct_L1_021: PASS
  greenwich_ct_U2_001: PASS
  bridgeport_ct_U2_003: PASS

STAGE 3A FINAL RESULT: PARTIAL — NOT READY FOR STAGE 3B

Meaning:
All results remain AI triage only. No verified zoning fact, legal conclusion,
candidate evidence record, human-review decision, subindex, composite, or
score has been produced.

CSV: outputs/triage/triage_stage3a_final_v1_2_20261002T180153Z.csv
JSON: outputs/triage/triage_stage3a_final_v1_2_20261002T180153Z.json
Summary: outputs/triage/triage_stage3

---

## Stage 3B — Limited U2 Candidate Extraction

This section is scoped to exactly two packets that passed Stage 3A triage/safeguards:
`greenwich_ct_U2_001` and `bridgeport_ct_U2_003`. It does **not** touch L1, does not
re-run Stage 3A, and does not process any other packet.

The output is two **candidate evidence records** — not verified zoning facts. Every
record is created with `extraction_status = candidate_text_found_needs_human_review`,
`score_eligible = false`, and `review_status = pending_review`, and is deterministically
validated in Python before being marked ready for human review. No automatic correction
of model output occurs — invalid or unsupported fields are preserved and flagged, not
silently fixed.

Exactly one Groq request is made per packet (two requests total this run).


In [1]:
# --- Stage 3B setup: load the two approved packets + their metadata only ---
# Self-contained: does not depend on earlier cells' in-memory state, so this section can
# run in isolation without re-executing (and re-calling the API from) prior Stage 3 cells.
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PILOT_ROOT = Path.cwd()
if not (PILOT_ROOT / "data").exists() and (PILOT_ROOT.parent / "zoning_pdf_pilot" / "data").exists():
    PILOT_ROOT = PILOT_ROOT.parent / "zoning_pdf_pilot"

load_dotenv(PILOT_ROOT.parent / ".env")
groq_api_key = os.environ.get("GROQ_API_KEY")
print(f"Groq key present: {bool(groq_api_key)}")

SELECTED_PACKET_IDS = ["greenwich_ct_U2_001", "bridgeport_ct_U2_003"]
MODEL_ID = "openai/gpt-oss-20b"
PROMPT_VERSION = "u2_extraction_v1"

all_packets = {p["packet_id"]: p for p in json.load(open(PILOT_ROOT / "data/processed/source_packets/source_packets_v1.json"))}
manifest_by_filename = {m["local_filename"]: m for m in json.load(open(PILOT_ROOT / "data/source_manifest/source_manifest_v1.json"))}

selected_packets = {pid: all_packets[pid] for pid in SELECTED_PACKET_IDS}
for pid, p in selected_packets.items():
    doc_hash = manifest_by_filename.get(p["source_filename"], {}).get("raw_pdf_sha256")
    print(f"{pid}: {p['source_filename']} (hash {doc_hash[:12]}...), pages {p['matched_pages']}")


Groq key present: True
greenwich_ct_U2_001: greenwich_ct.pdf (hash 18417c5e9074...), pages 13
bridgeport_ct_U2_003: bridgeport_ct.pdf (hash afe7c678fa5d...), pages 67-73


In [2]:
# --- Stage 3B: extraction prompt, schema, and deterministic validation ---
RULE_SCOPE_VALUES = [
    "base_district_candidate", "conditional_or_special_pathway_candidate", "building_type_candidate",
    "definition_only", "cross_reference_only", "procedure_only", "not_supported_by_packet", "unresolved",
]

EXTRACTION_INSTRUCTIONS = """You are creating a candidate evidence record from one public municipal
zoning source packet.

Your task is to reproduce what the supplied text supports for U2:
Two-Family Permission.

This is not a final legal conclusion. Do not infer missing facts from
outside knowledge. Do not guess. Use null when the packet does not
explicitly support a field.

A source quote must be copied exactly from the supplied packet. If no
single short quote supports the relevant claim, return source_quote = null.

Do not treat an ADU as two-family permission. Do not treat a definition
alone as permission. Do not infer a permission pathway from a district
label or building-type name alone.

For a form-based code, classify a building type as potentially relevant only
if the supplied text itself supports a two-unit/two-principal-unit
interpretation. If the evidence is incomplete, say so in uncertainty_notes.

Do not infer by-right permission unless the supplied packet explicitly says
permitted, allowed, by right, principal use, or uses an equivalent notation
whose meaning appears in the packet.
Do not infer special permit approval unless the supplied packet explicitly
identifies a special permit, special exception, discretionary review, or
equivalent pathway.
A district name is not proof of permission by itself.
For Bridgeport, classify "Double House A" as building_type_candidate only
if the packet text itself supports that it contains two principal housing
units or otherwise supports a two-unit interpretation.
If the source does not establish the approval pathway, set it to null.
If district applicability is unclear, set district_code and district_name
to null and explain the uncertainty.

Return exactly one valid JSON object and no other text."""

SCHEMA_TEXT = """{
  "packet_id": "string",
  "municipality": "string",
  "state": "string",
  "variable_id": "U2",
  "legal_subfield": "two_family_housing_permission",
  "district_code": null,
  "district_name": null,
  "building_type": null,
  "permission_status": null,
  "approval_pathway": null,
  "approval_body": null,
  "public_hearing_required": null,
  "rule_scope_classification": "base_district_candidate | conditional_or_special_pathway_candidate | building_type_candidate | definition_only | cross_reference_only | procedure_only | not_supported_by_packet | unresolved",
  "condition_text": null,
  "alternative_rules": [],
  "cross_reference_trail": [],
  "source_quote": null,
  "source_document_filename": "string",
  "source_document_hash": "string",
  "source_section_or_path": null,
  "source_table_title": null,
  "source_row_label": null,
  "source_column_label": null,
  "source_header_path": [],
  "source_footnote_text": null,
  "source_page_range": "string",
  "two_unit_interpretation_supported_by_packet": false,
  "two_unit_interpretation_reason": null,
  "uncertainty_notes": [],
  "ai_confidence": 0.0,
  "extraction_status": "candidate_text_found_needs_human_review",
  "score_eligible": false,
  "review_status": "pending_review"
}"""

NO_MARKDOWN_INSTRUCTION = """
Return exactly one valid JSON object.
Do not use Markdown.
Do not use a code fence.
Do not include explanation before or after the JSON.
Do not include reasoning."""


def build_extraction_prompt(packet, doc_hash):
    return f"""{EXTRACTION_INSTRUCTIONS}

PACKET METADATA:
packet_id: {packet['packet_id']}
municipality: {packet['municipality']}
state: CT
variable_id: U2
variable_name: Two-Family Permission
source_document_filename: {packet['source_filename']}
document_hash: {doc_hash}
matched_search_term: {packet['search_term_matched']}
page_range: {packet['matched_pages']}

PACKET TEXT:
{packet['extracted_text']}

Respond with a single JSON object matching exactly this shape (fill in real values, use null where not explicitly supported):
{SCHEMA_TEXT}
{NO_MARKDOWN_INSTRUCTION}"""


def normalize_ws(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip().lower()


REQUIRED_FIELDS = {
    "packet_id", "municipality", "state", "variable_id", "rule_scope_classification",
    "source_document_filename", "source_document_hash", "source_page_range",
    "two_unit_interpretation_supported_by_packet", "ai_confidence",
    "extraction_status", "score_eligible", "review_status",
}


def validate_candidate_record(parsed: dict, packet: dict, doc_hash: str) -> dict:
    """Deterministic validation only. Never silently edits model output — only flags."""
    issues = []

    missing = REQUIRED_FIELDS - set(parsed.keys())
    if missing:
        issues.append(f"missing_fields: {sorted(missing)}")

    if parsed.get("packet_id") != packet["packet_id"]:
        issues.append(f"packet_id_mismatch: {parsed.get('packet_id')!r}")
    if parsed.get("municipality") != packet["municipality"]:
        issues.append(f"municipality_mismatch: {parsed.get('municipality')!r}")
    if parsed.get("state") != "CT":
        issues.append(f"state_mismatch: {parsed.get('state')!r}")
    if parsed.get("variable_id") != "U2":
        issues.append(f"variable_id_mismatch: {parsed.get('variable_id')!r}")

    if parsed.get("rule_scope_classification") not in RULE_SCOPE_VALUES:
        issues.append(f"rule_scope_classification_not_permitted: {parsed.get('rule_scope_classification')!r}")

    conf = parsed.get("ai_confidence")
    if not (isinstance(conf, (int, float)) and 0.0 <= conf <= 1.0):
        issues.append(f"ai_confidence_invalid: {conf!r}")

    quote = parsed.get("source_quote")
    if quote and normalize_ws(quote) not in normalize_ws(packet["extracted_text"]):
        issues.append("source_quote_not_found_in_packet_text")

    if parsed.get("source_document_filename") != packet["source_filename"]:
        issues.append(f"source_document_filename_mismatch: {parsed.get('source_document_filename')!r}")
    if doc_hash and parsed.get("source_document_hash") != doc_hash:
        issues.append(f"source_document_hash_mismatch: {parsed.get('source_document_hash')!r}")
    if parsed.get("source_page_range") != packet["matched_pages"]:
        issues.append(f"source_page_range_mismatch: {parsed.get('source_page_range')!r}")

    if parsed.get("score_eligible") is not False:
        issues.append(f"score_eligible_not_false: {parsed.get('score_eligible')!r}")
    if parsed.get("review_status") != "pending_review":
        issues.append(f"review_status_not_pending: {parsed.get('review_status')!r}")
    if parsed.get("extraction_status") != "candidate_text_found_needs_human_review":
        issues.append(f"extraction_status_unexpected: {parsed.get('extraction_status')!r}")

    permission_status = parsed.get("permission_status")
    if permission_status:
        support_text = normalize_ws((quote or "") + " " + packet["extracted_text"])
        if not any(term in support_text for term in ("permit", "allow", "principal use", "by right")):
            issues.append("permission_status_set_without_clear_textual_support")

    approval_pathway = parsed.get("approval_pathway")
    if approval_pathway:
        support_text = normalize_ws((quote or "") + " " + packet["extracted_text"])
        if not any(term in support_text for term in ("special permit", "special exception", "discretionary", "review")):
            issues.append("approval_pathway_set_without_clear_textual_support")

    if parsed.get("two_unit_interpretation_supported_by_packet") is True:
        support_text = normalize_ws((quote or "") + " " + packet["extracted_text"])
        if not any(term in support_text for term in ("two-family", "two family", "multiple unit", "principal unit", "two unit", "two-unit")):
            issues.append("two_unit_interpretation_true_without_clear_textual_support")

    validation_status = "passed" if not issues else "failed"
    candidate_ready = validation_status == "passed"
    return {"validation_status": validation_status, "validation_issues": issues,
            "candidate_ready_for_human_review": candidate_ready}


print("Stage 3B prompt, schema, and validator defined (prompt version: u2_extraction_v1).")


Stage 3B prompt, schema, and validator defined (prompt version: u2_extraction_v1).


In [3]:
# --- Stage 3B: run exactly one Groq request per packet, validate, save ---
from groq import Groq

MAX_COMPLETION_TOKENS = 4096  # confirmed needed: Bridgeport's longer packet + reasoning
                               # tokens were cut off (finish_reason="length") at 2048.

stage3b_run_started_at = datetime.now(timezone.utc).isoformat()
stage3b_records = []

groq_client_3b = Groq(api_key=groq_api_key) if groq_api_key else None

for pid in SELECTED_PACKET_IDS:
    packet = selected_packets[pid]
    doc_hash = manifest_by_filename.get(packet["source_filename"], {}).get("raw_pdf_sha256")
    prompt = build_extraction_prompt(packet, doc_hash)
    request_ts = datetime.now(timezone.utc).isoformat()

    record = {
        "packet_id": pid, "municipality": packet["municipality"], "variable_id": "U2",
        "model_provider": "groq", "model_id": MODEL_ID, "prompt_version": PROMPT_VERSION,
        "request_timestamp_utc": request_ts, "raw_model_response": None,
        "parsed_candidate_record": None, "validation_status": None, "validation_issues": None,
        "candidate_ready_for_human_review": False, "provider_error": None,
    }

    if groq_client_3b is None:
        record["provider_error"] = "GROQ_API_KEY not present"
        stage3b_records.append(record)
        continue

    try:
        response = groq_client_3b.chat.completions.create(
            model=MODEL_ID,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_completion_tokens=MAX_COMPLETION_TOKENS,
        )
        content = response.choices[0].message.content or ""
        record["raw_model_response"] = content
        finish_reason = response.choices[0].finish_reason

        if not content.strip():
            record["provider_error"] = f"empty_completion (finish_reason={finish_reason})"
        else:
            try:
                parsed = json.loads(content)
                record["parsed_candidate_record"] = parsed
                validation = validate_candidate_record(parsed, packet, doc_hash)
                record["validation_status"] = validation["validation_status"]
                record["validation_issues"] = validation["validation_issues"]
                record["candidate_ready_for_human_review"] = validation["candidate_ready_for_human_review"]
            except json.JSONDecodeError as e:
                record["provider_error"] = f"non_json_completion (finish_reason={finish_reason}): {str(e)[:200]}"
    except Exception as exc:
        record["provider_error"] = f"{type(exc).__name__}: {str(exc)[:400]}"

    stage3b_records.append(record)

stage3b_run_finished_at = datetime.now(timezone.utc).isoformat()
print(f"Attempted {len(stage3b_records)} extractions.")
for r in stage3b_records:
    print(f"  {r['packet_id']}: validation_status={r['validation_status']}, "
          f"ready_for_review={r['candidate_ready_for_human_review']}, provider_error={r['provider_error']}")


Attempted 2 extractions.
  greenwich_ct_U2_001: validation_status=passed, ready_for_review=True, provider_error=None
  bridgeport_ct_U2_003: validation_status=passed, ready_for_review=True, provider_error=None


In [4]:
# --- Save Stage 3B outputs (new timestamped files only) and print the required summary ---
candidates_dir = PILOT_ROOT / "outputs" / "candidates"
stamp_3b = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
json_path_3b = candidates_dir / f"u2_extraction_batch_{stamp_3b}.json"
csv_path_3b = candidates_dir / f"u2_extraction_batch_{stamp_3b}.csv"
summary_path_3b = candidates_dir / f"u2_extraction_batch_{stamp_3b}_summary.json"

json_path_3b.write_text(json.dumps(stage3b_records, indent=2, default=str))

csv_rows = []
for r in stage3b_records:
    p = r.get("parsed_candidate_record") or {}
    csv_rows.append({
        "packet_id": r["packet_id"], "municipality": r["municipality"],
        "district_code": p.get("district_code"), "district_name": p.get("district_name"),
        "building_type": p.get("building_type"), "variable_id": r["variable_id"],
        "permission_status": p.get("permission_status"), "approval_pathway": p.get("approval_pathway"),
        "rule_scope_classification": p.get("rule_scope_classification"), "condition_text": p.get("condition_text"),
        "source_quote": p.get("source_quote"), "source_page_range": p.get("source_page_range"),
        "source_section_or_path": p.get("source_section_or_path"), "source_table_title": p.get("source_table_title"),
        "source_row_label": p.get("source_row_label"), "source_column_label": p.get("source_column_label"),
        "source_footnote_text": p.get("source_footnote_text"),
        "two_unit_interpretation_supported_by_packet": p.get("two_unit_interpretation_supported_by_packet"),
        "uncertainty_notes": p.get("uncertainty_notes"), "ai_confidence": p.get("ai_confidence"),
        "validation_status": r["validation_status"], "validation_issues": r["validation_issues"],
        "candidate_ready_for_human_review": r["candidate_ready_for_human_review"],
        "model_id": r["model_id"], "prompt_version": r["prompt_version"],
        "request_timestamp_utc": r["request_timestamp_utc"],
    })
pd.DataFrame(csv_rows).to_csv(csv_path_3b, index=False)

attempted = len(stage3b_records)
completed = sum(1 for r in stage3b_records if r["parsed_candidate_record"] is not None)
provider_errors = sum(1 for r in stage3b_records if r["provider_error"])
invalid_json = sum(1 for r in stage3b_records if r["provider_error"] and "non_json" in (r["provider_error"] or ""))
validation_passed = sum(1 for r in stage3b_records if r["validation_status"] == "passed")
validation_failed = sum(1 for r in stage3b_records if r["validation_status"] == "failed")
ready_count = sum(1 for r in stage3b_records if r["candidate_ready_for_human_review"])

if attempted == 0 or provider_errors == attempted:
    stage3b_status = "FAIL"
elif provider_errors > 0 or validation_failed > 0:
    stage3b_status = "PARTIAL"
else:
    stage3b_status = "PASS"

summary = {
    "selected_packet_count": len(SELECTED_PACKET_IDS), "attempted_packet_count": attempted,
    "completed_extraction_count": completed, "provider_error_count": provider_errors,
    "invalid_json_count": invalid_json, "validation_pass_count": validation_passed,
    "validation_fail_count": validation_failed, "candidate_ready_for_human_review_count": ready_count,
    "model_id": MODEL_ID, "prompt_version": PROMPT_VERSION,
    "run_started_at_utc": stage3b_run_started_at, "run_finished_at_utc": stage3b_run_finished_at,
    "stage_status": stage3b_status,
}
summary_path_3b.write_text(json.dumps(summary, indent=2))

by_pid = {r["packet_id"]: r for r in stage3b_records}

print("=" * 72)
print("ZONING PDF PILOT — STAGE 3B: LIMITED U2 CANDIDATE EXTRACTION")
print("=" * 72)
print(f"Selected packets: {len(SELECTED_PACKET_IDS)}")
print(f"Attempted extractions: {attempted}")
print(f"Completed model responses: {completed}")
print(f"Provider errors: {provider_errors}")
print(f"Invalid JSON responses: {invalid_json}")
print(f"Validation-passed candidate records: {validation_passed}")
print(f"Candidate records ready for human review: {ready_count}")
print()
for pid, label in [("greenwich_ct_U2_001", "Greenwich"), ("bridgeport_ct_U2_003", "Bridgeport")]:
    r = by_pid.get(pid, {})
    p = r.get("parsed_candidate_record") or {}
    print(f"{label} packet:")
    print(f"  ID: {pid}")
    print(f"  Validation: {r.get('validation_status')}")
    print(f"  Scope classification: {p.get('rule_scope_classification')}")
    print(f"  Candidate two-unit interpretation: {p.get('two_unit_interpretation_supported_by_packet')}")
    print()
print(f"Model provider: Groq")
print(f"Model ID: {MODEL_ID}")
print(f"Prompt version: {PROMPT_VERSION}")
print()
print(f"STAGE 3B RESULT: {stage3b_status}")
print()
print("Meaning:")
print("These are AI-generated candidate evidence records only. They are not verified")
print("zoning facts, legal conclusions, or score inputs. Human review against the")
print("original official PDF is required before any record can be accepted.")
print("=" * 72)
print()
print(f"JSON: {json_path_3b.relative_to(PILOT_ROOT)}")
print(f"CSV: {csv_path_3b.relative_to(PILOT_ROOT)}")
print(f"Summary: {summary_path_3b.relative_to(PILOT_ROOT)}")
print()
print("STOP: Stage 3B candidate extraction is complete. Human review has not started.")


ZONING PDF PILOT — STAGE 3B: LIMITED U2 CANDIDATE EXTRACTION
Selected packets: 2
Attempted extractions: 2
Completed model responses: 2
Provider errors: 0
Invalid JSON responses: 0
Validation-passed candidate records: 2
Candidate records ready for human review: 2

Greenwich packet:
  ID: greenwich_ct_U2_001
  Validation: passed
  Scope classification: definition_only
  Candidate two-unit interpretation: False

Bridgeport packet:
  ID: bridgeport_ct_U2_003
  Validation: passed
  Scope classification: building_type_candidate
  Candidate two-unit interpretation: True

Model provider: Groq
Model ID: openai/gpt-oss-20b
Prompt version: u2_extraction_v1

STAGE 3B RESULT: PASS

Meaning:
These are AI-generated candidate evidence records only. They are not verified
zoning facts, legal conclusions, or score inputs. Human review against the
original official PDF is required before any record can be accepted.

JSON: outputs/candidates/u2_extraction_batch_20261002T181737Z.json
CSV: outputs/candidates

---

## U2 Source Completion — Permission and Approval Context

Stage 3B's two U2 candidate records established relevant language exists for both
towns, but neither establishes a full permission picture:

- **Greenwich** (`greenwich_ct_U2_001`): "Single and Two-family Residence" is a district
  **name** in a zone-classification table. The packet does not show whether two-family
  use is by-right, conditional, or restricted in that district.
- **Bridgeport** (`bridgeport_ct_U2_003`): "Double House A" is described as a building
  type involving two House A types / multiple units, but the packet does not show
  *which zones* allow it or the approval pathway.

This section is **deterministic PDF search only** — no LLM call. It creates a small,
high-context supplement set of new source packets (distinct IDs, never overwriting the
original Stage 2 packets) aimed specifically at the missing permission/zone-applicability
context, for later human review and possible further extraction.


In [1]:
# --- U2 source completion: setup, term lists, manifest lookup (no LLM, self-contained) ---
import hashlib
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import pdfplumber

PILOT_ROOT = Path.cwd()
if not (PILOT_ROOT / "data").exists() and (PILOT_ROOT.parent / "zoning_pdf_pilot" / "data").exists():
    PILOT_ROOT = PILOT_ROOT.parent / "zoning_pdf_pilot"

manifest_by_filename = {m["local_filename"]: m for m in json.load(open(PILOT_ROOT / "data/source_manifest/source_manifest_v1.json"))}

GREENWICH_U2_TERMS = [
    "two-family", "two family", "two-family dwelling", "two family dwelling",
    "single and two-family", "r-6", "permitted uses", "use regulations", "use table",
    "special permit", "special exception", "site plan", "residence district", "dwelling",
]
BRIDGEPORT_U2_TERMS = [
    "double house a", "double house", "building type", "building types",
    "allowed building types", "permitted building types", "principal units",
    "allowed uses", "use table", "neighborhood zone", "nx1", "nx2", "special permit",
    "special exception", "administrative review", "site plan", "approval",
]
PRIORITY_SCORE_TERMS = [
    "permitted", "allowed", "principal use", "special permit", "special exception",
    "use table", "building type", "zone", "district", "approval",
]
EXCLUDE_IF_DOMINANT = ["section title", "effective\ndate", "effective date of amendment"]

MAX_NEW_PACKETS_PER_TOWN = 6
CONTEXT_PAGES = 1

print(f"Greenwich U2 follow-up search terms: {len(GREENWICH_U2_TERMS)}")
print(f"Bridgeport U2 follow-up search terms: {len(BRIDGEPORT_U2_TERMS)}")


Greenwich U2 follow-up search terms: 14
Bridgeport U2 follow-up search terms: 17


In [2]:
# --- Deterministic page search and context-window packet creation (no LLM) ---
MAX_PACKET_SPAN = 8  # hard cap: a "packet" spanning more than this many pages is not a
                       # tight, high-context supplement — it's merge-chaining gone wrong.


def sha256_text(text: str) -> str:
    return hashlib.sha256((text or "").encode("utf-8")).hexdigest()


def extract_all_page_texts(pdf_path: Path) -> list:
    with pdfplumber.open(pdf_path) as pdf:
        return [(page.extract_text() or "") for page in pdf.pages]


def find_term_hit_pages(page_texts: list, term: str) -> list:
    lowered = term.lower()
    return [i + 1 for i, text in enumerate(page_texts) if lowered in text.lower()]


def group_consecutive(pages: list) -> list:
    if not pages:
        return []
    pages = sorted(set(pages))
    groups, start, prev = [], pages[0], pages[0]
    for p in pages[1:]:
        if p == prev + 1:
            prev = p
            continue
        groups.append((start, prev))
        start, prev = p, p
    groups.append((start, prev))
    return groups


def shrink_to_best_window(page_range, page_texts, max_span):
    """If a single term's own consecutive run exceeds max_span, don't keep the whole
    run — find the page with the highest priority-term density inside it and return a
    tight window of width max_span centered there."""
    start, end = page_range
    if (end - start + 1) <= max_span:
        return page_range
    best_page, best_score = start, -1
    for page in range(start, end + 1):
        text = page_texts[page - 1].lower()
        score = sum(text.count(t) for t in PRIORITY_SCORE_TERMS)
        if score > best_score:
            best_score, best_page = score, page
    half = max_span // 2
    new_start = max(start, best_page - half)
    new_end = min(end, new_start + max_span - 1)
    return (new_start, new_end)


def ranges_overlap(a, b) -> bool:
    return a[0] <= b[1] and b[0] <= a[1]


def build_followup_packets(town_slug: str, municipality: str, pdf_filename: str, terms: list, id_prefix: str) -> list:
    pdf_path = PILOT_ROOT / "data/raw_pdfs" / town_slug / pdf_filename
    page_texts = extract_all_page_texts(pdf_path)
    doc_hash = manifest_by_filename.get(pdf_filename, {}).get("raw_pdf_sha256")

    # Build each term's hit-page runs, shrinking any run longer than MAX_PACKET_SPAN to
    # its highest-density window first — a single generic term matching many
    # consecutive pages should not produce one giant packet.
    raw_candidates = []
    for term in terms:
        hit_pages = find_term_hit_pages(page_texts, term)
        for pr in group_consecutive(hit_pages):
            raw_candidates.append((shrink_to_best_window(pr, page_texts, MAX_PACKET_SPAN), term))

    # Deduplicate identical/near-identical ranges first (e.g. "building type" vs
    # "building types" landing on the exact same window) by merging their term sets,
    # so near-synonym terms don't waste separate packet slots on the same content.
    by_range = {}
    for pr, term in raw_candidates:
        by_range.setdefault(pr, set()).add(term)
    deduped = [{"range": pr, "terms": terms} for pr, terms in by_range.items()]

    # Now merge only genuinely overlapping distinct ranges, still capped at MAX_PACKET_SPAN.
    merged = []
    for cand in sorted(deduped, key=lambda c: c["range"]):
        placed = False
        for m in merged:
            candidate_span = (min(m["range"][0], cand["range"][0]), max(m["range"][1], cand["range"][1]))
            if ranges_overlap(m["range"], cand["range"]) and (candidate_span[1] - candidate_span[0] + 1) <= MAX_PACKET_SPAN:
                m["range"] = candidate_span
                m["terms"] |= cand["terms"]
                placed = True
                break
        if not placed:
            merged.append({"range": cand["range"], "terms": set(cand["terms"])})

    packets = []
    for m in merged:
        start, end = m["range"]
        ctx_start = max(1, start - CONTEXT_PAGES)
        ctx_end = min(len(page_texts), end + CONTEXT_PAGES)
        combined_text = "\n\n".join(page_texts[ctx_start - 1:ctx_end])
        combined_text_norm = re.sub(r"[ \t]+", " ", combined_text).strip()

        lower_text = combined_text_norm.lower()
        priority_score = sum(lower_text.count(t) for t in PRIORITY_SCORE_TERMS)
        is_mostly_changelog = any(x in lower_text[:400] for x in EXCLUDE_IF_DOMINANT)

        packets.append({
            "page_range": f"{start}-{end}" if start != end else str(start),
            "context_page_range": f"{ctx_start}-{ctx_end}",
            "matched_search_terms": sorted(m["terms"]),
            "packet_text": combined_text_norm,
            "priority_score": priority_score,
            "excluded_as_changelog_only": is_mostly_changelog,
            "source_document_filename": pdf_filename,
            "source_document_hash": doc_hash,
            "municipality": municipality,
        })

    packets = [p for p in packets if not p["excluded_as_changelog_only"]]
    packets.sort(key=lambda p: p["priority_score"], reverse=True)
    selected = packets[:MAX_NEW_PACKETS_PER_TOWN]

    created_at = datetime.now(timezone.utc).isoformat()
    final_packets = []
    for i, p in enumerate(selected, start=1):
        final_packets.append({
            "packet_id": f"{id_prefix}_{i:03d}",
            "packet_stage": "u2_source_completion",
            "municipality": p["municipality"],
            "state": "CT",
            "variable_id": "U2",
            "variable_name": "Two-Family Permission",
            "source_document_filename": p["source_document_filename"],
            "source_document_hash": p["source_document_hash"],
            "page_range": p["page_range"],
            "matched_search_terms": p["matched_search_terms"],
            "section_heading_or_path": None,
            "table_context": None,
            "packet_text": p["packet_text"],
            "packet_text_hash": sha256_text(p["packet_text"]),
            "created_at_utc": created_at,
            "creation_method": "deterministic_pdf_search_with_context_pages",
            "packet_status": "unreviewed_candidate_source",
            "priority_score": p["priority_score"],
        })
    return final_packets


greenwich_followups = build_followup_packets(
    "greenwich_ct", "Greenwich", "greenwich_ct.pdf", GREENWICH_U2_TERMS, "greenwich_ct_U2_followup"
)
bridgeport_followups = build_followup_packets(
    "bridgeport_ct", "Bridgeport", "bridgeport_ct.pdf", BRIDGEPORT_U2_TERMS, "bridgeport_ct_U2_followup"
)

print(f"Greenwich: {len(greenwich_followups)} new packets")
for p in greenwich_followups:
    print(f"  {p['page_range']} score={p['priority_score']} terms={p['matched_search_terms']}")
print(f"Bridgeport: {len(bridgeport_followups)} new packets")
for p in bridgeport_followups:
    print(f"  {p['page_range']} score={p['priority_score']} terms={p['matched_search_terms']}")


Greenwich: 6 new packets
  80-87 score=129 terms=['dwelling', 'r-6', 'site plan', 'special permit', 'two family', 'two family dwelling', 'two-family', 'two-family dwelling', 'use regulations']
  101-107 score=117 terms=['dwelling', 'permitted uses', 'site plan', 'special exception', 'special permit', 'use regulations']
  95-100 score=107 terms=['dwelling', 'permitted uses', 'site plan', 'special exception', 'special permit', 'use regulations']
  4-7 score=93 terms=['dwelling', 'permitted uses', 'r-6', 'site plan', 'special permit', 'use regulations']
  113-118 score=93 terms=['dwelling', 'r-6', 'site plan', 'special permit', 'use regulations']
  130-132 score=93 terms=['site plan', 'special exception', 'special permit']
Bridgeport: 6 new packets
  12-19 score=205 terms=['allowed building types', 'allowed uses', 'approval', 'building type', 'building types', 'double house', 'double house a', 'neighborhood zone', 'nx1', 'nx2', 'special permit']
  264-271 score=131 terms=['site plan', 'sp

In [3]:
# --- Save new U2 source-completion packets (new timestamped files only) and print summary ---
all_followups = greenwich_followups + bridgeport_followups
stamp_u2fc = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

packets_dir = PILOT_ROOT / "data/processed/source_packets"
json_path_u2fc = packets_dir / f"u2_source_completion_packets_{stamp_u2fc}.json"
csv_path_u2fc = packets_dir / f"u2_source_completion_packets_{stamp_u2fc}.csv"
summary_path_u2fc = packets_dir / f"u2_source_completion_summary_{stamp_u2fc}.json"

json_path_u2fc.write_text(json.dumps(all_followups, indent=2))
pd.DataFrame(all_followups).drop(columns=["priority_score"]).to_csv(csv_path_u2fc, index=False)

summary = {
    "greenwich_new_packet_count": len(greenwich_followups),
    "bridgeport_new_packet_count": len(bridgeport_followups),
    "total_new_packet_count": len(all_followups),
    "greenwich_packet_ids": [p["packet_id"] for p in greenwich_followups],
    "bridgeport_packet_ids": [p["packet_id"] for p in bridgeport_followups],
    "llm_calls_made": 0,
    "created_at_utc": stamp_u2fc,
}
summary_path_u2fc.write_text(json.dumps(summary, indent=2))

result = "PASS" if (len(greenwich_followups) > 0 and len(bridgeport_followups) > 0) else "PARTIAL"

print("=" * 72)
print("ZONING PDF PILOT — U2 SOURCE COMPLETION RETRIEVAL")
print("=" * 72)
print()
print("Greenwich:")
print(f"  New U2 follow-up packets: {len(greenwich_followups)}")
print("  Pages/sections found:")
for p in greenwich_followups:
    preview = p["packet_text"][:80].replace("\n", " ")
    print(f"  - {p['packet_id']} | {p['page_range']} | {p['matched_search_terms']} | {preview}...")
print()
print("Bridgeport:")
print(f"  New U2 follow-up packets: {len(bridgeport_followups)}")
print("  Pages/sections found:")
for p in bridgeport_followups:
    preview = p["packet_text"][:80].replace("\n", " ")
    print(f"  - {p['packet_id']} | {p['page_range']} | {p['matched_search_terms']} | {preview}...")
print()
print(f"Total new packets: {len(all_followups)}")
print()
print("Meaning:")
print("These packets add use-table, zone, building-type, or approval-pathway")
print("context for later candidate extraction. They are not final zoning facts,")
print("do not establish legal permission by themselves, and have not been sent")
print("to an LLM.")
print()
print(f"RESULT: {result}")
print("=" * 72)
print()
print(f"JSON: {json_path_u2fc.relative_to(PILOT_ROOT)}")
print(f"CSV: {csv_path_u2fc.relative_to(PILOT_ROOT)}")
print(f"Summary: {summary_path_u2fc.relative_to(PILOT_ROOT)}")
print()
print("STOP: U2 source-completion packets are ready for review. No new U2 extraction has started.")


ZONING PDF PILOT — U2 SOURCE COMPLETION RETRIEVAL

Greenwich:
  New U2 follow-up packets: 6
  Pages/sections found:
  - greenwich_ct_U2_followup_001 | 80-87 | ['dwelling', 'r-6', 'site plan', 'special permit', 'two family', 'two family dwelling', 'two-family', 'two-family dwelling', 'use regulations'] | §6-94 GREENWICH MUNICIPAL CODE §6-94 shall be subject to review by the A.R.C. as...
  - greenwich_ct_U2_followup_002 | 101-107 | ['dwelling', 'permitted uses', 'site plan', 'special exception', 'special permit', 'use regulations'] | §6-103.1 GREENWICH MUNICIPAL CODE §6-103.1 wide and sub-regional market area; an...
  - greenwich_ct_U2_followup_003 | 95-100 | ['dwelling', 'permitted uses', 'site plan', 'special exception', 'special permit', 'use regulations'] | §6-100 GREENWICH MUNICIPAL CODE §6-100 offensive or obnoxious or detrimental to ...
  - greenwich_ct_U2_followup_004 | 4-7 | ['dwelling', 'permitted uses', 'r-6', 'site plan', 'special permit', 'use regulations'] | TABLE OF CONTEN